### 行情数据

In [ ]:
# -*- coding: utf-8 -*-
"""整份复制到 SuperMind JupyterHub 的一个单元格，一次执行（Python 3.8）。

前置条件：平台提供 pandas、已登录的研究 API；无需前序用户变量或已有文件。
沿用项目口径：沪深京历史 A 股、动态前复权、12列、聚宽风格代码。
仅直接导入 pandas 和平台 API。压缩通过 pandas 完成，不直接导入系统模块。
临时文件用 remove_file(trash=False) 彻底删除；只处理本次唯一名称。
正常异常/中断会执行 finally；内核被强杀时无法保证自动清理。
成品为标准 pandas 压缩 pickle，不再使用自定义 bundle 或 append。
"""

import pandas as pd

try:
    from mindgo_api import *
except ModuleNotFoundError:
    pass

START_DATE = "20260501"
END_DATE = "20260911"
FQ_MODE = "pre"
BATCH_SIZE = 100
SOURCE_FIELDS = [
    "open", "high", "low", "close", "prev_close",
    "volume", "turnover", "factor", "is_paused",
]
OUTPUT_COLUMNS = [
    "time", "code", "open", "high", "low", "close", "pre_close",
    "volume", "money", "factor", "paused", "return_1d",
]


def download_daily():
    # 在拉行情前直接引用所需 API，不使用 globals / getattr 等内省。
    required_apis = (get_trade_days, get_all_securities, get_price,
                     read_file, write_file, list_file, remove_file)
    tag = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
    prefix = "ths_daily_{}_{}_{}".format(START_DATE, END_DATE, tag)
    temp_name = prefix + "_temp.pkl"

    def clean_temp():
        if temp_name in list_file("", isfile=True):
            remove_file(temp_name, trash=False)
        if temp_name in list_file("", isfile=True):
            raise RuntimeError("本次临时文件删除失败：" + temp_name)

    def compress_and_check(frame, compression):
        try:
            frame.to_pickle(temp_name, compression=compression, protocol=4)
            restored = pd.read_pickle(temp_name, compression=compression["method"])
            if not restored.equals(frame) or restored.attrs != frame.attrs:
                raise RuntimeError("压缩回读与原数据不一致")
            del restored
            payload = read_file(temp_name)
            if not isinstance(payload, bytes) or not payload:
                raise RuntimeError("压缩文件未返回有效 bytes")
            return payload
        finally:
            clean_temp()

    print("请求：", START_DATE, "至", END_DATE, "；fq=", FQ_MODE)
    print("临时文件仅使用：", temp_name)
    # 先用小样本验证压缩与彻底清理，避免拉完数据才发现环境不支持。
    candidates = [
        ("xz", {"method": "xz", "preset": 6}),
        ("bz2", {"method": "bz2", "compresslevel": 9}),
        ("gz", {"method": "gzip", "compresslevel": 9}),
    ]
    available = []
    probe = pd.DataFrame({"code": ["000001.XSHE"], "close": [10.25]})
    for extension, compression in candidates:
        try:
            probe_bytes = compress_and_check(probe, compression)
        except (ImportError, ValueError, TypeError) as exc:
            print("此压缩方式不可用：", extension, str(exc))
        else:
            available.append((extension, compression))
            print("压缩及临时文件清理预检通过：", extension)
            del probe_bytes
    if not available:
        raise RuntimeError("没有通过预检的压缩方式，尚未拉取行情")
    del probe

    trade_days = pd.DatetimeIndex(get_trade_days(START_DATE, END_DATE)).normalize()
    if len(trade_days) == 0:
        raise RuntimeError("请求区间没有交易日")
    listing_dates = {}
    for number, day in enumerate(trade_days, 1):
        daily = get_all_securities("stock", day.strftime("%Y%m%d"))
        for symbol, listed in daily["listed_date"].items():
            symbol = str(symbol)
            if symbol.endswith((".SH", ".SZ", ".BJ")):
                listed = pd.Timestamp(listed).normalize()
                if symbol not in listing_dates or listed < listing_dates[symbol]:
                    listing_dates[symbol] = listed
        del daily
        if number % 10 == 0 or number == len(trade_days):
            print("证券范围 {}/{}，累计 {} 只".format(
                number, len(trade_days), len(listing_dates)))
    symbols = sorted(listing_dates)
    if not symbols:
        raise RuntimeError("历史 A 股证券列表为空")

    # 每批最多100只；行情仅在内存汇总，不产生行情中间文件。
    frames = []
    missing_symbols = []
    suffixes = {"SH": "XSHG", "SZ": "XSHE", "BJ": "XBSE"}
    for offset in range(0, len(symbols), BATCH_SIZE):
        batch = symbols[offset:offset + BATCH_SIZE]
        raw = get_price(batch, START_DATE, END_DATE, "1d", SOURCE_FIELDS,
                        skip_paused=False, fq=FQ_MODE, bar_count=0, is_panel=False)
        if isinstance(raw, pd.DataFrame) and len(batch) == 1:
            raw = {batch[0]: raw}
        if not isinstance(raw, dict):
            raise TypeError("多股票行情返回类型异常：" + type(raw).__name__)
        batch_rows = 0
        for symbol in batch:
            item = raw.get(symbol)
            if item is None or item.empty:
                missing_symbols.append(symbol)
                continue
            part = item[SOURCE_FIELDS].copy()
            part.index.name = "time"
            part = part.reset_index().rename(columns={
                "prev_close": "pre_close", "turnover": "money", "is_paused": "paused"})
            part["time"] = pd.to_datetime(part["time"]).dt.normalize()
            if not part["time"].isin(trade_days).all():
                raise RuntimeError("返回日期超出请求交易日范围：" + symbol)
            if part["time"].duplicated().any():
                raise RuntimeError("同一股票存在重复日线：" + symbol)
            if (part[["volume", "money"]] < 0).any().any():
                raise RuntimeError("成交量或成交额为负：" + symbol)
            code, exchange = symbol.split(".")
            part["code"] = code + "." + suffixes[exchange]
            valid = ((part["close"] > 0) & (part["pre_close"] > 0)
                     & (part["paused"].fillna(1) == 0)
                     & (part["time"] != listing_dates[symbol]))
            part["return_1d"] = (part["close"] / part["pre_close"] - 1).where(valid)
            frames.append(part[OUTPUT_COLUMNS])
            batch_rows += len(part)
            del part
        del raw, item
        print("行情批次 {}/{}，本批 {} 行，累计空返回 {} 只".format(
            offset // BATCH_SIZE + 1, (len(symbols) + BATCH_SIZE - 1) // BATCH_SIZE,
            batch_rows, len(missing_symbols)))
    if not frames:
        raise RuntimeError("所有股票行情为空，未生成成品")

    prices = pd.concat(frames, ignore_index=True)
    del frames
    prices = prices.sort_values(["code", "time"]).reset_index(drop=True)
    # 分类编码只压缩重复代码，不降低浮点精度，不删行情记录。
    prices["code"] = prices["code"].astype("category")
    missing_days = trade_days.difference(pd.DatetimeIndex(prices["time"].unique()))
    prices.attrs = {
        "source": "SuperMind", "start_date": START_DATE, "end_date": END_DATE,
        "fq": FQ_MODE, "frequency": "1d", "skip_paused": False,
        "created_at": pd.Timestamp.now().isoformat(),
        "universe": "union of daily Shanghai Shenzhen Beijing stock lists",
        "planned_symbols": symbols, "missing_symbols": missing_symbols,
        "missing_trade_days": [day.strftime("%Y%m%d") for day in missing_days],
        "field_mapping": "prev_close->pre_close; turnover->money; is_paused->paused",
        "code_mapping": ".SH->.XSHG; .SZ->.XSHE; .BJ->.XBSE",
        "money_unit": "CNY", "return_1d": "close/pre_close-1; null if paused/listing day/invalid price",
    }
    print("汇总：{} 行，{} 只，{} 个交易日，表内存 {:.2f} MiB".format(
        len(prices), prices["code"].nunique(), prices["time"].nunique(),
        prices.memory_usage(deep=True).sum() / 1024 ** 2))

    # 全表实测三种可用无损压缩，选字节数最小者；每次比较后立刻删临时文件。
    best_payload = None
    best_extension = None
    for extension, compression in available:
        print("正在压缩并回读核对：", extension)
        payload = compress_and_check(prices, compression)
        print("  {}：{:.3f} MiB".format(extension, len(payload) / 1024 ** 2))
        if best_payload is None or len(payload) < len(best_payload):
            best_payload = payload
            best_extension = extension
        del payload
    del prices

    # 全新文件名，完整内容一次写入；若超45MiB则保留标准压缩流的顺序分卷。
    output_name = prefix + ".pkl." + best_extension
    part_limit = 45 * 1024 ** 2
    total_bytes = len(best_payload)
    part_count = (total_bytes + part_limit - 1) // part_limit
    outputs = []
    for index in range(part_count):
        name = output_name if part_count == 1 else output_name + ".part{:03d}".format(index + 1)
        piece = best_payload[index * part_limit:(index + 1) * part_limit]
        try:
            written = write_file(name, piece)
            checked = read_file(name)
            if written != len(piece) or checked != piece:
                raise RuntimeError("成品写入核对失败：" + name)
        except BaseException:
            if name in list_file("", isfile=True):
                remove_file(name, trash=False)
            raise
        outputs.append(name)
        del piece, checked
    del best_payload
    clean_temp()
    print("生成完成；本次临时文件已彻底删除。")
    print("成品合计：{:.3f} MiB；请下载以下全部文件：".format(total_bytes / 1024 ** 2))
    for name in outputs:
        print(name)
    print("无行情返回的证券：", missing_symbols)
    print("全市场缺失交易日：", [day.strftime("%Y%m%d") for day in missing_days])
    if missing_symbols or len(missing_days):
        print("注意：包已生成，但存在上述覆盖缺口，不能视为全体行情完整。")
    print("平台曾拒绝3MiB文件，压缩成功不保证当日下载额度充足。")
    if part_count == 1:
        print("本地读取：prices = pd.read_pickle(r'下载目录/" + outputs[0] + "')")
    else:
        print("需按part序号拼接二进制为", output_name, "，再用pd.read_pickle读取。")
    return outputs


download_files = download_daily()


### 板块 成分股数据

In [ ]:
# -*- coding: utf-8 -*-
# 整份复制到 SuperMind 一个单元格执行，Python 3.8。
# 前置：平台已登录，研究 API 已预置；不依赖旧单元格变量或已有数据文件。
# 只直接导入 pandas。L2、L3 分别保存，避免多归属合并产生笛卡尔积。
import pandas as pd

START_DATE = "20260501"
END_DATE = "20260911"
MAX_FILE_BYTES = 45 * 1024 ** 2
DOWNLOAD_BUDGET_BYTES = 90 * 1024 ** 2  # 计划预算，不代表账号当前剩余额度


def remove_owned_file(path):
    if path in list_file("", isfile=True):
        remove_file(path, trash=False)
    if path in list_file("", isfile=True):
        raise RuntimeError("文件未彻底删除：" + path)


def compressed_bytes(frame, temp):
    if temp in list_file("", isfile=True):
        raise RuntimeError("临时名已存在，不覆盖：" + temp)
    try:
        frame.to_pickle(temp, compression={"method": "gzip", "compresslevel": 9}, protocol=4)
        restored = pd.read_pickle(temp, compression="gzip")
        if not restored.equals(frame) or restored.attrs != frame.attrs:
            raise RuntimeError("压缩回读不一致：" + temp)
        del restored
        payload = read_file(temp)
        if not isinstance(payload, bytes) or not payload:
            raise RuntimeError("压缩文件读取异常：" + temp)
        return payload
    finally:
        remove_owned_file(temp)


def write_checked(path, payload):
    if path in list_file("", isfile=True):
        raise RuntimeError("成品已存在，不覆盖：" + path)
    try:
        written = write_file(path, payload)
        if written != len(payload) or read_file(path) != payload:
            raise RuntimeError("成品字节核验失败：" + path)
    except BaseException:
        remove_owned_file(path)
        raise


def save_bounded(frame, prefix, level, limit, summary):
    # 每个分片都能直接 pd.read_pickle；按行顺序二分，不是二进制截断分卷。
    payload = compressed_bytes(frame, prefix + ".temp")
    size = len(payload)
    if size > limit:
        del payload
        if len(frame) <= 1:
            raise RuntimeError("单行及其元数据仍超过上限，请调大配置或检查目录元数据")
        middle = len(frame) // 2
        for start, end in ((0, middle), (middle, len(frame))):
            part = frame.iloc[start:end].copy().reset_index(drop=True)
            dates = sorted(part["time"].dt.strftime("%Y%m%d").unique().tolist())
            part.attrs = dict(frame.attrs)
            part.attrs["trade_dates"] = dates
            part.attrs["directories"] = {d: frame.attrs["directories"][d] for d in dates}
            part.attrs["coverage_checks"] = [
                item for item in frame.attrs["coverage_checks"] if item["date"] in dates
            ]
            for column in part.select_dtypes(include="category").columns:
                part[column] = part[column].cat.remove_unused_categories()
            save_bounded(part, prefix, level, limit, summary)
        return
    number = len(summary) + 1
    output = prefix + "_part{:03d}.pkl.gz".format(number)
    write_checked(output, payload)
    summary.append({"level": level, "part": number, "file": output,
                    "rows": len(frame), "bytes": size, "MiB": round(size / 1024 ** 2, 4)})
    print("已核验：{}，{}行，{}字节".format(output, len(frame), size))


def pull_l2_l3():
    required = (get_trade_days, get_industry_relate, get_industry_stocks,
                get_all_securities, read_file, write_file, list_file, remove_file)
    if MAX_FILE_BYTES <= 0 or DOWNLOAD_BUDGET_BYTES < MAX_FILE_BYTES:
        raise ValueError("大小上限必须为正，计划预算必须不小于单文件上限")
    tag = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
    prefix = "ths_industry_{}_{}_{}".format(START_DATE, END_DATE, tag)

    # 全量请求前确认当前内核的压缩、平台 bytes 写入、回读与彻底删除能力。
    probe_path = prefix + "_probe.pkl.gz"
    probe = pd.DataFrame({"symbol": ["000001.SZ"]})
    payload = compressed_bytes(probe, prefix + ".temp")
    write_checked(probe_path, payload)
    remove_owned_file(probe_path)
    del probe, payload

    days = pd.DatetimeIndex(get_trade_days(START_DATE, END_DATE)).normalize().sort_values()
    if days.empty or days.has_duplicates:
        raise RuntimeError("交易日历为空或重复")
    if days.min() < pd.Timestamp(START_DATE) or days.max() > pd.Timestamp(END_DATE):
        raise RuntimeError("交易日历超出请求范围")
    all_summary = []
    for level, industry_type in (("l2", "industryid2"), ("l3", "industryid3")):
        frames, directories, checks = [], {}, []
        for number, day in enumerate(days, 1):
            date = day.strftime("%Y%m%d")
            directory = get_industry_relate(date=date, types=industry_type, fields=None)
            if directory.empty or directory["industry_symbol"].isna().any():
                raise RuntimeError(level + " 目录为空或缺代码：" + date)
            if directory["industry_symbol"].duplicated().any():
                raise RuntimeError(level + " 目录代码重复：" + date)
            directories[date] = directory.reset_index().to_json(
                orient="records", force_ascii=False, date_format="iso")
            securities = get_all_securities("stock", date)
            if securities.empty or securities.index.has_duplicates:
                raise RuntimeError("股票目录为空或重复：" + date)
            stocks = {str(s) for s in securities.index if str(s).endswith((".SH", ".SZ", ".BJ"))}
            if not stocks:
                raise RuntimeError("沪深京股票范围为空：" + date)
            rows, empty_industries, outside = [], [], []
            for name, item in directory.iterrows():
                industry_code = str(item["industry_symbol"])
                members = get_industry_stocks(industry_code, date=date)
                if not isinstance(members, list) or len(members) != len(set(members)):
                    raise RuntimeError("成员返回异常或重复：{} {} {}".format(date, level, industry_code))
                if not members:
                    empty_industries.append(industry_code)
                for symbol in members:
                    symbol = str(symbol)
                    if symbol in stocks:
                        rows.append((symbol, industry_code, str(name)))
                    else:
                        outside.append((symbol, industry_code))
            daily = pd.DataFrame(rows, columns=["symbol", level + "_code", level + "_name"])
            missing = sorted(stocks.difference(daily["symbol"]))
            multiple = sorted(daily.loc[daily["symbol"].duplicated(keep=False), "symbol"].unique().tolist())
            if missing:
                daily = pd.concat([daily, pd.DataFrame({"symbol": missing})], ignore_index=True)
            daily.insert(0, "time", day)
            for column in ["symbol", level + "_code", level + "_name"]:
                daily[column] = daily[column].astype("category")
            frames.append(daily)
            checks.append({"date": date, "stock_count": len(stocks), "industry_count": len(directory),
                           "empty_industries": empty_industries, "empty_status": "empty_unverified",
                           "missing_symbols": missing, "multiple_symbols": multiple,
                           "outside_stock_universe": outside})
            print("{} {}/{} {}：{}个行业，{}行，未映射{}只，多归属{}只，空行业{}个，范围外关系{}条".format(
                level.upper(), number, len(days), date, len(directory), len(daily),
                len(missing), len(multiple), len(empty_industries), len(outside)))
            del daily, rows, securities, directory, members

        result = pd.concat(frames, ignore_index=True)
        del frames
        result["code"] = (result["symbol"].astype(str)
                          .str.replace(".SH", ".XSHG", regex=False)
                          .str.replace(".SZ", ".XSHE", regex=False)
                          .str.replace(".BJ", ".XBSE", regex=False))
        result = result[["time", "code", "symbol", level + "_code", level + "_name"]]
        for column in result.columns[1:]:
            result[column] = result[column].astype("category")
        if pd.DatetimeIndex(result["time"].unique()).sort_values().tolist() != days.tolist():
            raise RuntimeError(level + " 结果日期未覆盖全部交易日")
        result.attrs = {"source": "SuperMind", "start_date": START_DATE, "end_date": END_DATE,
                        "trade_dates": [d.strftime("%Y%m%d") for d in days],
                        "industry_type": industry_type, "directories": directories,
                        "coverage_checks": checks, "created_at": pd.Timestamp.now().isoformat()}
        summary = []
        try:
            save_bounded(result, prefix + "_" + level, level, MAX_FILE_BYTES, summary)
            if sum(item["rows"] for item in summary) != len(result):
                raise RuntimeError("分片行数不一致")
        except BaseException:
            # 仅清理本次尚未完成层级的成品；已经完成的另一层级保留。
            for item in summary:
                remove_owned_file(item["file"])
            raise
        all_summary.extend(summary)
        del result, directories, checks
        print(level.upper() + " 生成完成，所有文件回读一致、大小合格。")

    delivery = pd.DataFrame(all_summary)
    batch, used, batches = 1, 0, []
    for size in delivery["bytes"]:
        if used + size > DOWNLOAD_BUDGET_BYTES:
            batch, used = batch + 1, 0
        batches.append(batch)
        used += size
    delivery["planned_download_batch"] = batches
    print(delivery.to_string(index=False))
    print("L2和L3均生成完成：{}个文件，总计{}字节（{:.3f}MiB）。".format(
        len(delivery), int(delivery["bytes"].sum()), delivery["bytes"].sum() / 1024 ** 2))
    print("按planned_download_batch分批下载；当前账号剩余额度未知，实际下载成功需另行确认。")
    print("每片可直接pd.read_pickle读取；同层级按part顺序pd.concat合并。")
    print("缺失、多归属、空返回和范围外关系见每片.attrs['coverage_checks']；目录见.attrs['directories']。")
    return delivery


download_summary = pull_l2_l3()

### 财务数据

In [ ]:
# -*- coding: utf-8 -*-
"""整份复制到SuperMind Python 3.8 Notebook的一个单元格执行。
前置：已登录，内核预置query、get_fundamentals、balance、income及文件API。
股票清单已从指定行情成品嵌入，无需上传行情文件或执行其他单元格。
按已实测整表接口拉取；未验证字段属性，不猜测字段裁剪调用。
仅拉2025Q4、2026Q1、2026Q2；income_sq为平台单季度利润表，不作累计相减。
财报版本为latest=False的当前返回，不保证历史全部更正版本。
只保存原始财务记录，不在此计算五因子或回填至每日行情。
"""
import pandas as pd

SYMBOLS = """000001.SZ
000002.SZ
000004.SZ
000006.SZ
000007.SZ
000008.SZ
000009.SZ
000010.SZ
000011.SZ
000012.SZ
000014.SZ
000016.SZ
000017.SZ
000019.SZ
000020.SZ
000021.SZ
000025.SZ
000026.SZ
000027.SZ
000028.SZ
000029.SZ
000030.SZ
000031.SZ
000032.SZ
000034.SZ
000035.SZ
000036.SZ
000037.SZ
000039.SZ
000042.SZ
000045.SZ
000048.SZ
000049.SZ
000050.SZ
000055.SZ
000056.SZ
000058.SZ
000059.SZ
000060.SZ
000061.SZ
000062.SZ
000063.SZ
000065.SZ
000066.SZ
000068.SZ
000069.SZ
000070.SZ
000078.SZ
000088.SZ
000089.SZ
000090.SZ
000096.SZ
000099.SZ
000100.SZ
000151.SZ
000153.SZ
000155.SZ
000156.SZ
000157.SZ
000158.SZ
000159.SZ
000166.SZ
000301.SZ
000333.SZ
000338.SZ
000400.SZ
000401.SZ
000402.SZ
000403.SZ
000404.SZ
000407.SZ
000408.SZ
000409.SZ
000410.SZ
000411.SZ
000415.SZ
000417.SZ
000419.SZ
000420.SZ
000421.SZ
000422.SZ
000423.SZ
000425.SZ
000426.SZ
000428.SZ
000429.SZ
000430.SZ
000488.SZ
000498.SZ
000501.SZ
000503.SZ
000504.SZ
000505.SZ
000506.SZ
000507.SZ
000509.SZ
000510.SZ
000513.SZ
000514.SZ
000516.SZ
000517.SZ
000518.SZ
000519.SZ
000520.SZ
000521.SZ
000523.SZ
000524.SZ
000525.SZ
000526.SZ
000528.SZ
000529.SZ
000530.SZ
000531.SZ
000532.SZ
000533.SZ
000534.SZ
000536.SZ
000537.SZ
000538.SZ
000539.SZ
000541.SZ
000543.SZ
000544.SZ
000545.SZ
000546.SZ
000547.SZ
000548.SZ
000550.SZ
000551.SZ
000552.SZ
000553.SZ
000554.SZ
000555.SZ
000557.SZ
000558.SZ
000559.SZ
000560.SZ
000561.SZ
000563.SZ
000564.SZ
000565.SZ
000566.SZ
000567.SZ
000568.SZ
000570.SZ
000571.SZ
000572.SZ
000573.SZ
000576.SZ
000581.SZ
000582.SZ
000586.SZ
000589.SZ
000590.SZ
000591.SZ
000592.SZ
000593.SZ
000595.SZ
000596.SZ
000597.SZ
000598.SZ
000599.SZ
000600.SZ
000601.SZ
000603.SZ
000605.SZ
000607.SZ
000608.SZ
000609.SZ
000610.SZ
000612.SZ
000615.SZ
000617.SZ
000619.SZ
000620.SZ
000623.SZ
000625.SZ
000626.SZ
000628.SZ
000629.SZ
000630.SZ
000631.SZ
000632.SZ
000633.SZ
000635.SZ
000636.SZ
000637.SZ
000638.SZ
000639.SZ
000650.SZ
000651.SZ
000652.SZ
000655.SZ
000656.SZ
000657.SZ
000659.SZ
000661.SZ
000663.SZ
000665.SZ
000668.SZ
000669.SZ
000670.SZ
000672.SZ
000676.SZ
000677.SZ
000678.SZ
000679.SZ
000680.SZ
000681.SZ
000682.SZ
000683.SZ
000685.SZ
000686.SZ
000688.SZ
000690.SZ
000691.SZ
000692.SZ
000695.SZ
000697.SZ
000698.SZ
000700.SZ
000701.SZ
000702.SZ
000703.SZ
000705.SZ
000707.SZ
000708.SZ
000709.SZ
000710.SZ
000711.SZ
000712.SZ
000713.SZ
000715.SZ
000716.SZ
000717.SZ
000718.SZ
000719.SZ
000720.SZ
000721.SZ
000722.SZ
000723.SZ
000725.SZ
000726.SZ
000727.SZ
000728.SZ
000729.SZ
000731.SZ
000733.SZ
000735.SZ
000736.SZ
000737.SZ
000738.SZ
000739.SZ
000750.SZ
000751.SZ
000752.SZ
000753.SZ
000755.SZ
000756.SZ
000757.SZ
000758.SZ
000759.SZ
000761.SZ
000762.SZ
000766.SZ
000767.SZ
000768.SZ
000776.SZ
000777.SZ
000778.SZ
000779.SZ
000782.SZ
000783.SZ
000785.SZ
000786.SZ
000788.SZ
000789.SZ
000790.SZ
000791.SZ
000792.SZ
000793.SZ
000795.SZ
000796.SZ
000797.SZ
000798.SZ
000799.SZ
000800.SZ
000801.SZ
000802.SZ
000803.SZ
000807.SZ
000809.SZ
000810.SZ
000811.SZ
000812.SZ
000813.SZ
000815.SZ
000816.SZ
000818.SZ
000819.SZ
000820.SZ
000821.SZ
000822.SZ
000823.SZ
000825.SZ
000826.SZ
000828.SZ
000829.SZ
000830.SZ
000831.SZ
000833.SZ
000837.SZ
000838.SZ
000839.SZ
000848.SZ
000850.SZ
000852.SZ
000856.SZ
000858.SZ
000859.SZ
000860.SZ
000862.SZ
000863.SZ
000868.SZ
000869.SZ
000875.SZ
000876.SZ
000877.SZ
000878.SZ
000880.SZ
000881.SZ
000882.SZ
000883.SZ
000885.SZ
000886.SZ
000887.SZ
000888.SZ
000889.SZ
000890.SZ
000892.SZ
000893.SZ
000895.SZ
000897.SZ
000898.SZ
000899.SZ
000900.SZ
000901.SZ
000902.SZ
000903.SZ
000905.SZ
000906.SZ
000908.SZ
000909.SZ
000910.SZ
000911.SZ
000912.SZ
000913.SZ
000915.SZ
000917.SZ
000919.SZ
000920.SZ
000921.SZ
000922.SZ
000923.SZ
000925.SZ
000926.SZ
000927.SZ
000928.SZ
000929.SZ
000930.SZ
000931.SZ
000932.SZ
000933.SZ
000935.SZ
000936.SZ
000937.SZ
000938.SZ
000948.SZ
000949.SZ
000950.SZ
000951.SZ
000952.SZ
000953.SZ
000955.SZ
000957.SZ
000958.SZ
000959.SZ
000960.SZ
000962.SZ
000963.SZ
000965.SZ
000966.SZ
000967.SZ
000968.SZ
000969.SZ
000970.SZ
000972.SZ
000973.SZ
000975.SZ
000977.SZ
000978.SZ
000980.SZ
000981.SZ
000983.SZ
000985.SZ
000987.SZ
000988.SZ
000989.SZ
000990.SZ
000993.SZ
000995.SZ
000997.SZ
000998.SZ
000999.SZ
001201.SZ
001202.SZ
001203.SZ
001205.SZ
001206.SZ
001207.SZ
001208.SZ
001209.SZ
001210.SZ
001211.SZ
001212.SZ
001213.SZ
001215.SZ
001216.SZ
001217.SZ
001218.SZ
001219.SZ
001220.SZ
001221.SZ
001222.SZ
001223.SZ
001225.SZ
001226.SZ
001227.SZ
001228.SZ
001229.SZ
001230.SZ
001231.SZ
001232.SZ
001233.SZ
001234.SZ
001236.SZ
001237.SZ
001238.SZ
001239.SZ
001248.SZ
001255.SZ
001256.SZ
001257.SZ
001258.SZ
001259.SZ
001260.SZ
001266.SZ
001267.SZ
001268.SZ
001269.SZ
001270.SZ
001277.SZ
001278.SZ
001279.SZ
001280.SZ
001282.SZ
001283.SZ
001285.SZ
001286.SZ
001287.SZ
001288.SZ
001289.SZ
001296.SZ
001298.SZ
001299.SZ
001300.SZ
001301.SZ
001306.SZ
001308.SZ
001309.SZ
001311.SZ
001312.SZ
001313.SZ
001314.SZ
001316.SZ
001317.SZ
001318.SZ
001319.SZ
001322.SZ
001323.SZ
001324.SZ
001325.SZ
001326.SZ
001328.SZ
001330.SZ
001331.SZ
001332.SZ
001333.SZ
001335.SZ
001336.SZ
001337.SZ
001338.SZ
001339.SZ
001356.SZ
001358.SZ
001359.SZ
001360.SZ
001365.SZ
001366.SZ
001367.SZ
001368.SZ
001369.SZ
001373.SZ
001376.SZ
001378.SZ
001379.SZ
001380.SZ
001382.SZ
001386.SZ
001387.SZ
001388.SZ
001389.SZ
001390.SZ
001391.SZ
001393.SZ
001395.SZ
001396.SZ
001399.SZ
001400.SZ
001696.SZ
001872.SZ
001896.SZ
001914.SZ
001965.SZ
001979.SZ
002001.SZ
002003.SZ
002004.SZ
002005.SZ
002006.SZ
002007.SZ
002008.SZ
002009.SZ
002010.SZ
002011.SZ
002012.SZ
002014.SZ
002015.SZ
002016.SZ
002017.SZ
002019.SZ
002020.SZ
002021.SZ
002022.SZ
002023.SZ
002024.SZ
002025.SZ
002026.SZ
002027.SZ
002028.SZ
002029.SZ
002030.SZ
002031.SZ
002032.SZ
002033.SZ
002034.SZ
002035.SZ
002036.SZ
002037.SZ
002038.SZ
002039.SZ
002040.SZ
002041.SZ
002042.SZ
002043.SZ
002044.SZ
002045.SZ
002046.SZ
002047.SZ
002048.SZ
002049.SZ
002050.SZ
002051.SZ
002052.SZ
002053.SZ
002054.SZ
002055.SZ
002056.SZ
002057.SZ
002058.SZ
002059.SZ
002060.SZ
002061.SZ
002062.SZ
002063.SZ
002064.SZ
002065.SZ
002066.SZ
002067.SZ
002068.SZ
002069.SZ
002072.SZ
002073.SZ
002074.SZ
002075.SZ
002076.SZ
002077.SZ
002078.SZ
002079.SZ
002080.SZ
002081.SZ
002082.SZ
002083.SZ
002084.SZ
002085.SZ
002086.SZ
002088.SZ
002090.SZ
002091.SZ
002092.SZ
002093.SZ
002094.SZ
002095.SZ
002096.SZ
002097.SZ
002098.SZ
002099.SZ
002100.SZ
002101.SZ
002102.SZ
002103.SZ
002104.SZ
002105.SZ
002106.SZ
002107.SZ
002108.SZ
002109.SZ
002110.SZ
002111.SZ
002112.SZ
002114.SZ
002115.SZ
002116.SZ
002117.SZ
002119.SZ
002120.SZ
002121.SZ
002122.SZ
002123.SZ
002124.SZ
002125.SZ
002126.SZ
002127.SZ
002128.SZ
002129.SZ
002130.SZ
002131.SZ
002132.SZ
002133.SZ
002134.SZ
002135.SZ
002136.SZ
002137.SZ
002138.SZ
002139.SZ
002140.SZ
002141.SZ
002142.SZ
002144.SZ
002145.SZ
002146.SZ
002148.SZ
002149.SZ
002150.SZ
002151.SZ
002152.SZ
002153.SZ
002154.SZ
002155.SZ
002156.SZ
002157.SZ
002158.SZ
002159.SZ
002160.SZ
002161.SZ
002162.SZ
002163.SZ
002164.SZ
002165.SZ
002166.SZ
002167.SZ
002168.SZ
002169.SZ
002170.SZ
002171.SZ
002172.SZ
002173.SZ
002174.SZ
002175.SZ
002176.SZ
002177.SZ
002178.SZ
002179.SZ
002180.SZ
002181.SZ
002182.SZ
002183.SZ
002184.SZ
002185.SZ
002186.SZ
002187.SZ
002188.SZ
002189.SZ
002190.SZ
002191.SZ
002192.SZ
002193.SZ
002194.SZ
002195.SZ
002196.SZ
002197.SZ
002198.SZ
002199.SZ
002200.SZ
002201.SZ
002202.SZ
002203.SZ
002204.SZ
002205.SZ
002206.SZ
002207.SZ
002208.SZ
002209.SZ
002210.SZ
002211.SZ
002212.SZ
002213.SZ
002214.SZ
002215.SZ
002216.SZ
002217.SZ
002218.SZ
002219.SZ
002221.SZ
002222.SZ
002223.SZ
002224.SZ
002225.SZ
002226.SZ
002227.SZ
002228.SZ
002229.SZ
002230.SZ
002232.SZ
002233.SZ
002234.SZ
002235.SZ
002236.SZ
002237.SZ
002238.SZ
002239.SZ
002240.SZ
002241.SZ
002242.SZ
002243.SZ
002244.SZ
002245.SZ
002246.SZ
002247.SZ
002248.SZ
002249.SZ
002250.SZ
002251.SZ
002252.SZ
002253.SZ
002254.SZ
002255.SZ
002256.SZ
002258.SZ
002259.SZ
002261.SZ
002262.SZ
002263.SZ
002264.SZ
002265.SZ
002266.SZ
002267.SZ
002268.SZ
002269.SZ
002270.SZ
002271.SZ
002272.SZ
002273.SZ
002274.SZ
002275.SZ
002276.SZ
002277.SZ
002278.SZ
002279.SZ
002281.SZ
002282.SZ
002283.SZ
002284.SZ
002285.SZ
002286.SZ
002287.SZ
002289.SZ
002290.SZ
002291.SZ
002292.SZ
002293.SZ
002294.SZ
002295.SZ
002296.SZ
002297.SZ
002298.SZ
002299.SZ
002300.SZ
002301.SZ
002302.SZ
002303.SZ
002304.SZ
002305.SZ
002306.SZ
002307.SZ
002309.SZ
002310.SZ
002311.SZ
002312.SZ
002313.SZ
002314.SZ
002315.SZ
002316.SZ
002317.SZ
002318.SZ
002319.SZ
002320.SZ
002321.SZ
002322.SZ
002323.SZ
002324.SZ
002326.SZ
002327.SZ
002328.SZ
002329.SZ
002330.SZ
002331.SZ
002332.SZ
002333.SZ
002334.SZ
002335.SZ
002337.SZ
002338.SZ
002339.SZ
002340.SZ
002342.SZ
002343.SZ
002344.SZ
002345.SZ
002346.SZ
002347.SZ
002348.SZ
002349.SZ
002350.SZ
002351.SZ
002352.SZ
002353.SZ
002354.SZ
002355.SZ
002356.SZ
002357.SZ
002358.SZ
002360.SZ
002361.SZ
002362.SZ
002363.SZ
002364.SZ
002365.SZ
002366.SZ
002367.SZ
002368.SZ
002369.SZ
002370.SZ
002371.SZ
002372.SZ
002373.SZ
002374.SZ
002375.SZ
002376.SZ
002377.SZ
002378.SZ
002379.SZ
002380.SZ
002381.SZ
002382.SZ
002383.SZ
002384.SZ
002385.SZ
002386.SZ
002387.SZ
002388.SZ
002389.SZ
002390.SZ
002391.SZ
002392.SZ
002393.SZ
002394.SZ
002395.SZ
002396.SZ
002397.SZ
002398.SZ
002399.SZ
002400.SZ
002401.SZ
002402.SZ
002403.SZ
002404.SZ
002405.SZ
002406.SZ
002407.SZ
002408.SZ
002409.SZ
002410.SZ
002412.SZ
002413.SZ
002414.SZ
002415.SZ
002416.SZ
002418.SZ
002419.SZ
002420.SZ
002421.SZ
002422.SZ
002423.SZ
002424.SZ
002425.SZ
002426.SZ
002427.SZ
002428.SZ
002429.SZ
002430.SZ
002431.SZ
002432.SZ
002434.SZ
002436.SZ
002437.SZ
002438.SZ
002439.SZ
002440.SZ
002441.SZ
002442.SZ
002443.SZ
002444.SZ
002445.SZ
002446.SZ
002448.SZ
002449.SZ
002451.SZ
002452.SZ
002453.SZ
002454.SZ
002455.SZ
002456.SZ
002457.SZ
002458.SZ
002459.SZ
002460.SZ
002461.SZ
002462.SZ
002463.SZ
002465.SZ
002466.SZ
002467.SZ
002468.SZ
002469.SZ
002470.SZ
002471.SZ
002472.SZ
002474.SZ
002475.SZ
002476.SZ
002478.SZ
002479.SZ
002480.SZ
002481.SZ
002482.SZ
002483.SZ
002484.SZ
002485.SZ
002486.SZ
002487.SZ
002488.SZ
002489.SZ
002490.SZ
002491.SZ
002492.SZ
002493.SZ
002494.SZ
002495.SZ
002496.SZ
002497.SZ
002498.SZ
002500.SZ
002501.SZ
002506.SZ
002507.SZ
002508.SZ
002510.SZ
002511.SZ
002512.SZ
002513.SZ
002514.SZ
002515.SZ
002516.SZ
002517.SZ
002518.SZ
002519.SZ
002520.SZ
002521.SZ
002522.SZ
002523.SZ
002524.SZ
002526.SZ
002527.SZ
002528.SZ
002529.SZ
002530.SZ
002531.SZ
002532.SZ
002533.SZ
002534.SZ
002535.SZ
002536.SZ
002537.SZ
002538.SZ
002539.SZ
002540.SZ
002541.SZ
002542.SZ
002543.SZ
002544.SZ
002545.SZ
002546.SZ
002547.SZ
002548.SZ
002549.SZ
002550.SZ
002551.SZ
002552.SZ
002553.SZ
002554.SZ
002555.SZ
002556.SZ
002557.SZ
002558.SZ
002559.SZ
002560.SZ
002561.SZ
002562.SZ
002563.SZ
002564.SZ
002565.SZ
002566.SZ
002567.SZ
002568.SZ
002569.SZ
002570.SZ
002571.SZ
002572.SZ
002573.SZ
002574.SZ
002575.SZ
002576.SZ
002577.SZ
002578.SZ
002579.SZ
002580.SZ
002581.SZ
002582.SZ
002583.SZ
002584.SZ
002585.SZ
002586.SZ
002587.SZ
002588.SZ
002589.SZ
002590.SZ
002591.SZ
002592.SZ
002593.SZ
002594.SZ
002595.SZ
002596.SZ
002597.SZ
002598.SZ
002599.SZ
002600.SZ
002601.SZ
002602.SZ
002603.SZ
002605.SZ
002606.SZ
002607.SZ
002608.SZ
002609.SZ
002611.SZ
002612.SZ
002613.SZ
002614.SZ
002615.SZ
002616.SZ
002617.SZ
002620.SZ
002622.SZ
002623.SZ
002624.SZ
002625.SZ
002626.SZ
002627.SZ
002628.SZ
002629.SZ
002630.SZ
002631.SZ
002632.SZ
002633.SZ
002634.SZ
002635.SZ
002636.SZ
002637.SZ
002638.SZ
002639.SZ
002640.SZ
002641.SZ
002642.SZ
002643.SZ
002644.SZ
002645.SZ
002646.SZ
002647.SZ
002648.SZ
002649.SZ
002650.SZ
002651.SZ
002652.SZ
002653.SZ
002654.SZ
002655.SZ
002656.SZ
002657.SZ
002658.SZ
002659.SZ
002660.SZ
002661.SZ
002662.SZ
002663.SZ
002664.SZ
002666.SZ
002667.SZ
002668.SZ
002669.SZ
002670.SZ
002671.SZ
002672.SZ
002673.SZ
002674.SZ
002675.SZ
002676.SZ
002677.SZ
002678.SZ
002679.SZ
002681.SZ
002682.SZ
002683.SZ
002685.SZ
002686.SZ
002687.SZ
002688.SZ
002689.SZ
002690.SZ
002691.SZ
002692.SZ
002693.SZ
002694.SZ
002695.SZ
002696.SZ
002697.SZ
002698.SZ
002700.SZ
002701.SZ
002702.SZ
002703.SZ
002705.SZ
002706.SZ
002707.SZ
002708.SZ
002709.SZ
002712.SZ
002713.SZ
002714.SZ
002715.SZ
002716.SZ
002717.SZ
002718.SZ
002719.SZ
002721.SZ
002722.SZ
002723.SZ
002724.SZ
002725.SZ
002726.SZ
002727.SZ
002728.SZ
002729.SZ
002730.SZ
002731.SZ
002732.SZ
002733.SZ
002734.SZ
002735.SZ
002736.SZ
002737.SZ
002738.SZ
002739.SZ
002741.SZ
002742.SZ
002743.SZ
002745.SZ
002746.SZ
002747.SZ
002748.SZ
002749.SZ
002752.SZ
002753.SZ
002755.SZ
002756.SZ
002757.SZ
002758.SZ
002759.SZ
002760.SZ
002761.SZ
002762.SZ
002763.SZ
002765.SZ
002766.SZ
002767.SZ
002768.SZ
002769.SZ
002771.SZ
002772.SZ
002773.SZ
002774.SZ
002775.SZ
002777.SZ
002778.SZ
002779.SZ
002780.SZ
002782.SZ
002783.SZ
002785.SZ
002786.SZ
002787.SZ
002788.SZ
002789.SZ
002790.SZ
002791.SZ
002792.SZ
002793.SZ
002795.SZ
002796.SZ
002797.SZ
002798.SZ
002799.SZ
002800.SZ
002801.SZ
002802.SZ
002803.SZ
002805.SZ
002806.SZ
002807.SZ
002808.SZ
002809.SZ
002810.SZ
002811.SZ
002812.SZ
002813.SZ
002815.SZ
002816.SZ
002817.SZ
002818.SZ
002819.SZ
002820.SZ
002821.SZ
002822.SZ
002823.SZ
002824.SZ
002825.SZ
002826.SZ
002827.SZ
002828.SZ
002829.SZ
002830.SZ
002831.SZ
002832.SZ
002833.SZ
002835.SZ
002836.SZ
002837.SZ
002838.SZ
002839.SZ
002840.SZ
002841.SZ
002842.SZ
002843.SZ
002845.SZ
002846.SZ
002847.SZ
002848.SZ
002849.SZ
002850.SZ
002851.SZ
002852.SZ
002853.SZ
002855.SZ
002856.SZ
002857.SZ
002858.SZ
002859.SZ
002860.SZ
002861.SZ
002862.SZ
002863.SZ
002864.SZ
002865.SZ
002866.SZ
002867.SZ
002868.SZ
002869.SZ
002870.SZ
002871.SZ
002872.SZ
002873.SZ
002875.SZ
002876.SZ
002877.SZ
002878.SZ
002879.SZ
002880.SZ
002881.SZ
002882.SZ
002883.SZ
002884.SZ
002885.SZ
002886.SZ
002887.SZ
002888.SZ
002889.SZ
002890.SZ
002891.SZ
002892.SZ
002893.SZ
002895.SZ
002896.SZ
002897.SZ
002898.SZ
002899.SZ
002900.SZ
002901.SZ
002902.SZ
002903.SZ
002905.SZ
002906.SZ
002907.SZ
002908.SZ
002909.SZ
002910.SZ
002911.SZ
002912.SZ
002913.SZ
002915.SZ
002916.SZ
002917.SZ
002918.SZ
002919.SZ
002920.SZ
002921.SZ
002922.SZ
002923.SZ
002925.SZ
002926.SZ
002927.SZ
002928.SZ
002929.SZ
002930.SZ
002931.SZ
002932.SZ
002933.SZ
002935.SZ
002936.SZ
002937.SZ
002938.SZ
002939.SZ
002940.SZ
002941.SZ
002942.SZ
002943.SZ
002945.SZ
002946.SZ
002947.SZ
002948.SZ
002949.SZ
002950.SZ
002951.SZ
002952.SZ
002953.SZ
002955.SZ
002956.SZ
002957.SZ
002958.SZ
002959.SZ
002960.SZ
002961.SZ
002962.SZ
002963.SZ
002965.SZ
002966.SZ
002967.SZ
002968.SZ
002969.SZ
002970.SZ
002971.SZ
002972.SZ
002973.SZ
002975.SZ
002976.SZ
002977.SZ
002978.SZ
002979.SZ
002980.SZ
002981.SZ
002982.SZ
002983.SZ
002984.SZ
002985.SZ
002986.SZ
002987.SZ
002988.SZ
002989.SZ
002990.SZ
002991.SZ
002992.SZ
002993.SZ
002995.SZ
002996.SZ
002997.SZ
002998.SZ
002999.SZ
003000.SZ
003001.SZ
003002.SZ
003003.SZ
003004.SZ
003005.SZ
003006.SZ
003007.SZ
003008.SZ
003009.SZ
003010.SZ
003011.SZ
003012.SZ
003013.SZ
003015.SZ
003016.SZ
003017.SZ
003018.SZ
003019.SZ
003020.SZ
003021.SZ
003022.SZ
003023.SZ
003025.SZ
003026.SZ
003027.SZ
003028.SZ
003029.SZ
003030.SZ
003031.SZ
003032.SZ
003033.SZ
003035.SZ
003036.SZ
003037.SZ
003038.SZ
003039.SZ
003040.SZ
003041.SZ
003042.SZ
003043.SZ
003816.SZ
300001.SZ
300002.SZ
300003.SZ
300004.SZ
300005.SZ
300006.SZ
300007.SZ
300008.SZ
300009.SZ
300010.SZ
300011.SZ
300012.SZ
300013.SZ
300014.SZ
300015.SZ
300016.SZ
300017.SZ
300018.SZ
300019.SZ
300020.SZ
300021.SZ
300022.SZ
300024.SZ
300025.SZ
300026.SZ
300027.SZ
300029.SZ
300030.SZ
300031.SZ
300032.SZ
300033.SZ
300034.SZ
300035.SZ
300036.SZ
300037.SZ
300039.SZ
300040.SZ
300041.SZ
300042.SZ
300043.SZ
300044.SZ
300045.SZ
300046.SZ
300047.SZ
300048.SZ
300049.SZ
300050.SZ
300051.SZ
300052.SZ
300053.SZ
300054.SZ
300055.SZ
300056.SZ
300057.SZ
300058.SZ
300059.SZ
300061.SZ
300062.SZ
300063.SZ
300065.SZ
300066.SZ
300067.SZ
300068.SZ
300069.SZ
300070.SZ
300071.SZ
300072.SZ
300073.SZ
300074.SZ
300075.SZ
300076.SZ
300077.SZ
300078.SZ
300079.SZ
300080.SZ
300081.SZ
300082.SZ
300083.SZ
300084.SZ
300085.SZ
300086.SZ
300087.SZ
300088.SZ
300091.SZ
300092.SZ
300093.SZ
300094.SZ
300095.SZ
300096.SZ
300097.SZ
300098.SZ
300099.SZ
300100.SZ
300101.SZ
300102.SZ
300103.SZ
300105.SZ
300106.SZ
300107.SZ
300109.SZ
300110.SZ
300111.SZ
300112.SZ
300113.SZ
300115.SZ
300118.SZ
300119.SZ
300120.SZ
300121.SZ
300122.SZ
300123.SZ
300124.SZ
300125.SZ
300126.SZ
300127.SZ
300128.SZ
300129.SZ
300130.SZ
300131.SZ
300132.SZ
300133.SZ
300134.SZ
300135.SZ
300136.SZ
300137.SZ
300138.SZ
300139.SZ
300140.SZ
300141.SZ
300142.SZ
300143.SZ
300144.SZ
300145.SZ
300146.SZ
300147.SZ
300148.SZ
300149.SZ
300150.SZ
300151.SZ
300152.SZ
300153.SZ
300154.SZ
300155.SZ
300157.SZ
300158.SZ
300159.SZ
300160.SZ
300161.SZ
300162.SZ
300163.SZ
300164.SZ
300165.SZ
300166.SZ
300167.SZ
300168.SZ
300169.SZ
300170.SZ
300171.SZ
300172.SZ
300173.SZ
300174.SZ
300175.SZ
300176.SZ
300177.SZ
300179.SZ
300180.SZ
300181.SZ
300182.SZ
300183.SZ
300184.SZ
300185.SZ
300187.SZ
300188.SZ
300189.SZ
300190.SZ
300191.SZ
300192.SZ
300193.SZ
300194.SZ
300195.SZ
300196.SZ
300197.SZ
300198.SZ
300199.SZ
300200.SZ
300201.SZ
300203.SZ
300204.SZ
300205.SZ
300206.SZ
300207.SZ
300209.SZ
300210.SZ
300211.SZ
300212.SZ
300213.SZ
300214.SZ
300215.SZ
300217.SZ
300218.SZ
300219.SZ
300220.SZ
300221.SZ
300222.SZ
300223.SZ
300224.SZ
300225.SZ
300226.SZ
300227.SZ
300228.SZ
300229.SZ
300230.SZ
300231.SZ
300232.SZ
300233.SZ
300234.SZ
300235.SZ
300236.SZ
300237.SZ
300238.SZ
300239.SZ
300240.SZ
300241.SZ
300242.SZ
300243.SZ
300244.SZ
300245.SZ
300246.SZ
300247.SZ
300248.SZ
300249.SZ
300250.SZ
300251.SZ
300252.SZ
300253.SZ
300254.SZ
300255.SZ
300256.SZ
300257.SZ
300258.SZ
300259.SZ
300260.SZ
300261.SZ
300263.SZ
300264.SZ
300265.SZ
300266.SZ
300267.SZ
300268.SZ
300269.SZ
300270.SZ
300271.SZ
300272.SZ
300274.SZ
300275.SZ
300276.SZ
300277.SZ
300278.SZ
300279.SZ
300281.SZ
300283.SZ
300284.SZ
300285.SZ
300286.SZ
300287.SZ
300288.SZ
300289.SZ
300290.SZ
300291.SZ
300292.SZ
300293.SZ
300294.SZ
300295.SZ
300296.SZ
300298.SZ
300299.SZ
300300.SZ
300301.SZ
300302.SZ
300303.SZ
300304.SZ
300305.SZ
300306.SZ
300307.SZ
300308.SZ
300310.SZ
300311.SZ
300313.SZ
300314.SZ
300315.SZ
300316.SZ
300317.SZ
300318.SZ
300319.SZ
300320.SZ
300321.SZ
300322.SZ
300323.SZ
300324.SZ
300326.SZ
300327.SZ
300328.SZ
300329.SZ
300331.SZ
300332.SZ
300333.SZ
300334.SZ
300335.SZ
300337.SZ
300338.SZ
300339.SZ
300340.SZ
300341.SZ
300342.SZ
300343.SZ
300345.SZ
300346.SZ
300347.SZ
300348.SZ
300349.SZ
300350.SZ
300351.SZ
300352.SZ
300353.SZ
300354.SZ
300355.SZ
300357.SZ
300358.SZ
300359.SZ
300360.SZ
300363.SZ
300364.SZ
300365.SZ
300366.SZ
300368.SZ
300369.SZ
300370.SZ
300371.SZ
300373.SZ
300374.SZ
300375.SZ
300376.SZ
300377.SZ
300378.SZ
300380.SZ
300381.SZ
300382.SZ
300383.SZ
300384.SZ
300385.SZ
300386.SZ
300387.SZ
300388.SZ
300389.SZ
300390.SZ
300393.SZ
300394.SZ
300395.SZ
300396.SZ
300397.SZ
300398.SZ
300399.SZ
300400.SZ
300401.SZ
300402.SZ
300403.SZ
300404.SZ
300405.SZ
300406.SZ
300407.SZ
300408.SZ
300409.SZ
300410.SZ
300411.SZ
300412.SZ
300413.SZ
300414.SZ
300415.SZ
300416.SZ
300417.SZ
300418.SZ
300419.SZ
300420.SZ
300421.SZ
300422.SZ
300423.SZ
300424.SZ
300425.SZ
300426.SZ
300427.SZ
300428.SZ
300429.SZ
300430.SZ
300432.SZ
300433.SZ
300434.SZ
300435.SZ
300436.SZ
300437.SZ
300438.SZ
300439.SZ
300440.SZ
300441.SZ
300442.SZ
300443.SZ
300444.SZ
300445.SZ
300446.SZ
300447.SZ
300448.SZ
300449.SZ
300450.SZ
300451.SZ
300452.SZ
300453.SZ
300454.SZ
300455.SZ
300456.SZ
300457.SZ
300458.SZ
300459.SZ
300460.SZ
300461.SZ
300462.SZ
300463.SZ
300464.SZ
300465.SZ
300466.SZ
300467.SZ
300468.SZ
300469.SZ
300470.SZ
300471.SZ
300472.SZ
300473.SZ
300474.SZ
300475.SZ
300476.SZ
300477.SZ
300478.SZ
300479.SZ
300480.SZ
300481.SZ
300482.SZ
300483.SZ
300484.SZ
300485.SZ
300486.SZ
300487.SZ
300488.SZ
300489.SZ
300490.SZ
300491.SZ
300492.SZ
300493.SZ
300494.SZ
300496.SZ
300497.SZ
300498.SZ
300499.SZ
300500.SZ
300501.SZ
300502.SZ
300503.SZ
300504.SZ
300505.SZ
300506.SZ
300507.SZ
300508.SZ
300509.SZ
300510.SZ
300511.SZ
300512.SZ
300513.SZ
300514.SZ
300515.SZ
300516.SZ
300517.SZ
300518.SZ
300519.SZ
300520.SZ
300521.SZ
300522.SZ
300523.SZ
300525.SZ
300527.SZ
300528.SZ
300529.SZ
300530.SZ
300531.SZ
300532.SZ
300533.SZ
300534.SZ
300535.SZ
300536.SZ
300537.SZ
300538.SZ
300539.SZ
300540.SZ
300541.SZ
300542.SZ
300543.SZ
300545.SZ
300546.SZ
300547.SZ
300548.SZ
300549.SZ
300550.SZ
300551.SZ
300552.SZ
300553.SZ
300554.SZ
300555.SZ
300556.SZ
300557.SZ
300558.SZ
300559.SZ
300560.SZ
300561.SZ
300562.SZ
300563.SZ
300564.SZ
300565.SZ
300566.SZ
300567.SZ
300568.SZ
300569.SZ
300570.SZ
300571.SZ
300572.SZ
300573.SZ
300575.SZ
300576.SZ
300577.SZ
300578.SZ
300579.SZ
300580.SZ
300581.SZ
300582.SZ
300583.SZ
300584.SZ
300585.SZ
300586.SZ
300587.SZ
300588.SZ
300589.SZ
300590.SZ
300591.SZ
300592.SZ
300593.SZ
300594.SZ
300595.SZ
300596.SZ
300597.SZ
300598.SZ
300599.SZ
300600.SZ
300601.SZ
300602.SZ
300603.SZ
300604.SZ
300605.SZ
300606.SZ
300607.SZ
300608.SZ
300609.SZ
300610.SZ
300611.SZ
300612.SZ
300613.SZ
300614.SZ
300615.SZ
300616.SZ
300617.SZ
300618.SZ
300619.SZ
300620.SZ
300621.SZ
300622.SZ
300623.SZ
300624.SZ
300625.SZ
300626.SZ
300627.SZ
300628.SZ
300629.SZ
300631.SZ
300632.SZ
300633.SZ
300634.SZ
300635.SZ
300636.SZ
300637.SZ
300638.SZ
300639.SZ
300640.SZ
300641.SZ
300642.SZ
300643.SZ
300644.SZ
300645.SZ
300647.SZ
300648.SZ
300649.SZ
300650.SZ
300651.SZ
300652.SZ
300653.SZ
300654.SZ
300655.SZ
300656.SZ
300657.SZ
300658.SZ
300659.SZ
300660.SZ
300661.SZ
300662.SZ
300663.SZ
300664.SZ
300665.SZ
300666.SZ
300667.SZ
300668.SZ
300669.SZ
300670.SZ
300671.SZ
300672.SZ
300673.SZ
300674.SZ
300675.SZ
300676.SZ
300677.SZ
300678.SZ
300679.SZ
300680.SZ
300681.SZ
300682.SZ
300683.SZ
300684.SZ
300685.SZ
300686.SZ
300687.SZ
300688.SZ
300689.SZ
300690.SZ
300691.SZ
300692.SZ
300693.SZ
300694.SZ
300695.SZ
300696.SZ
300697.SZ
300698.SZ
300699.SZ
300700.SZ
300701.SZ
300702.SZ
300703.SZ
300705.SZ
300706.SZ
300707.SZ
300708.SZ
300709.SZ
300710.SZ
300711.SZ
300712.SZ
300713.SZ
300715.SZ
300716.SZ
300717.SZ
300718.SZ
300719.SZ
300720.SZ
300721.SZ
300722.SZ
300723.SZ
300724.SZ
300725.SZ
300726.SZ
300727.SZ
300729.SZ
300730.SZ
300731.SZ
300732.SZ
300733.SZ
300735.SZ
300736.SZ
300737.SZ
300738.SZ
300739.SZ
300740.SZ
300741.SZ
300743.SZ
300745.SZ
300746.SZ
300747.SZ
300748.SZ
300749.SZ
300750.SZ
300751.SZ
300752.SZ
300753.SZ
300755.SZ
300756.SZ
300757.SZ
300758.SZ
300759.SZ
300760.SZ
300761.SZ
300762.SZ
300763.SZ
300765.SZ
300766.SZ
300767.SZ
300768.SZ
300769.SZ
300770.SZ
300771.SZ
300772.SZ
300773.SZ
300774.SZ
300775.SZ
300776.SZ
300777.SZ
300778.SZ
300779.SZ
300780.SZ
300781.SZ
300782.SZ
300783.SZ
300784.SZ
300785.SZ
300786.SZ
300787.SZ
300788.SZ
300789.SZ
300790.SZ
300791.SZ
300792.SZ
300793.SZ
300795.SZ
300796.SZ
300797.SZ
300798.SZ
300800.SZ
300801.SZ
300802.SZ
300803.SZ
300804.SZ
300805.SZ
300806.SZ
300807.SZ
300808.SZ
300809.SZ
300810.SZ
300811.SZ
300812.SZ
300813.SZ
300814.SZ
300815.SZ
300816.SZ
300817.SZ
300818.SZ
300819.SZ
300820.SZ
300821.SZ
300822.SZ
300823.SZ
300824.SZ
300825.SZ
300826.SZ
300827.SZ
300828.SZ
300829.SZ
300830.SZ
300831.SZ
300832.SZ
300833.SZ
300834.SZ
300835.SZ
300836.SZ
300837.SZ
300838.SZ
300839.SZ
300840.SZ
300841.SZ
300842.SZ
300843.SZ
300844.SZ
300845.SZ
300846.SZ
300847.SZ
300848.SZ
300849.SZ
300850.SZ
300851.SZ
300852.SZ
300853.SZ
300854.SZ
300855.SZ
300856.SZ
300857.SZ
300858.SZ
300859.SZ
300860.SZ
300861.SZ
300862.SZ
300863.SZ
300864.SZ
300865.SZ
300866.SZ
300867.SZ
300868.SZ
300869.SZ
300870.SZ
300871.SZ
300872.SZ
300873.SZ
300875.SZ
300876.SZ
300877.SZ
300878.SZ
300879.SZ
300880.SZ
300881.SZ
300882.SZ
300883.SZ
300884.SZ
300885.SZ
300886.SZ
300887.SZ
300888.SZ
300889.SZ
300890.SZ
300891.SZ
300892.SZ
300893.SZ
300894.SZ
300895.SZ
300896.SZ
300897.SZ
300898.SZ
300899.SZ
300900.SZ
300901.SZ
300902.SZ
300903.SZ
300904.SZ
300905.SZ
300906.SZ
300907.SZ
300908.SZ
300909.SZ
300910.SZ
300911.SZ
300912.SZ
300913.SZ
300915.SZ
300916.SZ
300917.SZ
300918.SZ
300919.SZ
300920.SZ
300921.SZ
300922.SZ
300923.SZ
300925.SZ
300926.SZ
300927.SZ
300928.SZ
300929.SZ
300930.SZ
300931.SZ
300932.SZ
300933.SZ
300935.SZ
300936.SZ
300937.SZ
300938.SZ
300939.SZ
300940.SZ
300941.SZ
300942.SZ
300943.SZ
300945.SZ
300946.SZ
300947.SZ
300948.SZ
300949.SZ
300950.SZ
300951.SZ
300952.SZ
300953.SZ
300955.SZ
300956.SZ
300957.SZ
300958.SZ
300959.SZ
300960.SZ
300961.SZ
300962.SZ
300963.SZ
300964.SZ
300965.SZ
300966.SZ
300967.SZ
300968.SZ
300969.SZ
300970.SZ
300971.SZ
300972.SZ
300973.SZ
300975.SZ
300976.SZ
300977.SZ
300978.SZ
300979.SZ
300980.SZ
300981.SZ
300982.SZ
300983.SZ
300984.SZ
300985.SZ
300986.SZ
300987.SZ
300988.SZ
300989.SZ
300990.SZ
300991.SZ
300992.SZ
300993.SZ
300994.SZ
300995.SZ
300996.SZ
300997.SZ
300998.SZ
300999.SZ
301000.SZ
301001.SZ
301002.SZ
301003.SZ
301004.SZ
301005.SZ
301006.SZ
301007.SZ
301008.SZ
301009.SZ
301010.SZ
301011.SZ
301012.SZ
301013.SZ
301015.SZ
301016.SZ
301017.SZ
301018.SZ
301019.SZ
301020.SZ
301021.SZ
301022.SZ
301023.SZ
301024.SZ
301025.SZ
301026.SZ
301027.SZ
301028.SZ
301029.SZ
301030.SZ
301031.SZ
301032.SZ
301033.SZ
301035.SZ
301036.SZ
301037.SZ
301038.SZ
301039.SZ
301040.SZ
301041.SZ
301042.SZ
301043.SZ
301045.SZ
301046.SZ
301047.SZ
301048.SZ
301049.SZ
301050.SZ
301051.SZ
301052.SZ
301053.SZ
301055.SZ
301056.SZ
301057.SZ
301058.SZ
301059.SZ
301060.SZ
301061.SZ
301062.SZ
301063.SZ
301065.SZ
301066.SZ
301067.SZ
301068.SZ
301069.SZ
301070.SZ
301071.SZ
301072.SZ
301073.SZ
301075.SZ
301076.SZ
301077.SZ
301078.SZ
301079.SZ
301080.SZ
301081.SZ
301082.SZ
301083.SZ
301085.SZ
301086.SZ
301087.SZ
301088.SZ
301089.SZ
301090.SZ
301091.SZ
301092.SZ
301093.SZ
301095.SZ
301096.SZ
301097.SZ
301098.SZ
301099.SZ
301100.SZ
301101.SZ
301102.SZ
301103.SZ
301105.SZ
301106.SZ
301107.SZ
301108.SZ
301109.SZ
301110.SZ
301111.SZ
301112.SZ
301113.SZ
301115.SZ
301116.SZ
301117.SZ
301118.SZ
301119.SZ
301120.SZ
301121.SZ
301122.SZ
301123.SZ
301125.SZ
301126.SZ
301127.SZ
301128.SZ
301129.SZ
301130.SZ
301131.SZ
301132.SZ
301133.SZ
301135.SZ
301136.SZ
301137.SZ
301138.SZ
301139.SZ
301141.SZ
301148.SZ
301149.SZ
301150.SZ
301151.SZ
301152.SZ
301153.SZ
301155.SZ
301156.SZ
301157.SZ
301158.SZ
301159.SZ
301160.SZ
301161.SZ
301162.SZ
301163.SZ
301165.SZ
301166.SZ
301167.SZ
301168.SZ
301169.SZ
301170.SZ
301171.SZ
301172.SZ
301173.SZ
301175.SZ
301176.SZ
301177.SZ
301178.SZ
301179.SZ
301180.SZ
301181.SZ
301182.SZ
301183.SZ
301185.SZ
301186.SZ
301187.SZ
301188.SZ
301189.SZ
301190.SZ
301191.SZ
301192.SZ
301193.SZ
301195.SZ
301196.SZ
301197.SZ
301198.SZ
301199.SZ
301200.SZ
301201.SZ
301202.SZ
301203.SZ
301205.SZ
301206.SZ
301207.SZ
301208.SZ
301209.SZ
301210.SZ
301211.SZ
301212.SZ
301213.SZ
301215.SZ
301216.SZ
301217.SZ
301218.SZ
301219.SZ
301220.SZ
301221.SZ
301222.SZ
301223.SZ
301225.SZ
301226.SZ
301227.SZ
301228.SZ
301229.SZ
301230.SZ
301231.SZ
301232.SZ
301233.SZ
301234.SZ
301235.SZ
301236.SZ
301237.SZ
301238.SZ
301239.SZ
301246.SZ
301248.SZ
301251.SZ
301252.SZ
301255.SZ
301256.SZ
301257.SZ
301258.SZ
301259.SZ
301260.SZ
301261.SZ
301262.SZ
301263.SZ
301265.SZ
301266.SZ
301267.SZ
301268.SZ
301269.SZ
301270.SZ
301272.SZ
301273.SZ
301275.SZ
301276.SZ
301277.SZ
301278.SZ
301279.SZ
301280.SZ
301281.SZ
301282.SZ
301283.SZ
301285.SZ
301286.SZ
301287.SZ
301288.SZ
301289.SZ
301290.SZ
301291.SZ
301292.SZ
301293.SZ
301295.SZ
301296.SZ
301297.SZ
301298.SZ
301299.SZ
301300.SZ
301301.SZ
301302.SZ
301303.SZ
301305.SZ
301306.SZ
301307.SZ
301308.SZ
301309.SZ
301310.SZ
301311.SZ
301312.SZ
301313.SZ
301314.SZ
301315.SZ
301316.SZ
301317.SZ
301318.SZ
301319.SZ
301320.SZ
301321.SZ
301322.SZ
301323.SZ
301325.SZ
301326.SZ
301327.SZ
301328.SZ
301329.SZ
301330.SZ
301331.SZ
301332.SZ
301333.SZ
301335.SZ
301336.SZ
301337.SZ
301338.SZ
301339.SZ
301345.SZ
301348.SZ
301349.SZ
301353.SZ
301355.SZ
301356.SZ
301357.SZ
301358.SZ
301359.SZ
301360.SZ
301361.SZ
301362.SZ
301363.SZ
301365.SZ
301366.SZ
301367.SZ
301368.SZ
301369.SZ
301370.SZ
301371.SZ
301372.SZ
301373.SZ
301376.SZ
301377.SZ
301378.SZ
301379.SZ
301380.SZ
301381.SZ
301382.SZ
301383.SZ
301386.SZ
301387.SZ
301388.SZ
301389.SZ
301390.SZ
301391.SZ
301392.SZ
301393.SZ
301395.SZ
301396.SZ
301397.SZ
301398.SZ
301399.SZ
301408.SZ
301413.SZ
301418.SZ
301419.SZ
301421.SZ
301428.SZ
301429.SZ
301439.SZ
301446.SZ
301448.SZ
301449.SZ
301456.SZ
301458.SZ
301459.SZ
301468.SZ
301469.SZ
301479.SZ
301486.SZ
301487.SZ
301488.SZ
301489.SZ
301491.SZ
301498.SZ
301499.SZ
301500.SZ
301501.SZ
301502.SZ
301503.SZ
301505.SZ
301507.SZ
301508.SZ
301509.SZ
301510.SZ
301511.SZ
301512.SZ
301513.SZ
301515.SZ
301516.SZ
301517.SZ
301518.SZ
301519.SZ
301520.SZ
301522.SZ
301525.SZ
301526.SZ
301528.SZ
301529.SZ
301531.SZ
301533.SZ
301535.SZ
301536.SZ
301538.SZ
301539.SZ
301548.SZ
301550.SZ
301551.SZ
301552.SZ
301555.SZ
301556.SZ
301557.SZ
301558.SZ
301559.SZ
301560.SZ
301563.SZ
301565.SZ
301566.SZ
301567.SZ
301568.SZ
301571.SZ
301575.SZ
301577.SZ
301578.SZ
301580.SZ
301581.SZ
301583.SZ
301584.SZ
301585.SZ
301586.SZ
301587.SZ
301588.SZ
301589.SZ
301590.SZ
301591.SZ
301592.SZ
301595.SZ
301596.SZ
301598.SZ
301599.SZ
301600.SZ
301601.SZ
301602.SZ
301603.SZ
301606.SZ
301607.SZ
301608.SZ
301609.SZ
301611.SZ
301613.SZ
301616.SZ
301617.SZ
301618.SZ
301622.SZ
301626.SZ
301628.SZ
301629.SZ
301630.SZ
301631.SZ
301632.SZ
301633.SZ
301636.SZ
301638.SZ
301655.SZ
301656.SZ
301658.SZ
301662.SZ
301665.SZ
301666.SZ
301667.SZ
301668.SZ
301669.SZ
301677.SZ
301678.SZ
301680.SZ
301682.SZ
301683.SZ
301687.SZ
301688.SZ
301689.SZ
301696.SZ
301697.SZ
301699.SZ
301707.SZ
301717.SZ
302132.SZ
430017.BJ
430047.BJ
430090.BJ
430139.BJ
430198.BJ
430300.BJ
430418.BJ
430425.BJ
430476.BJ
430478.BJ
430489.BJ
430510.BJ
430556.BJ
430564.BJ
430685.BJ
430718.BJ
600000.SH
600004.SH
600006.SH
600007.SH
600008.SH
600009.SH
600010.SH
600011.SH
600012.SH
600015.SH
600016.SH
600017.SH
600018.SH
600019.SH
600020.SH
600021.SH
600022.SH
600023.SH
600025.SH
600026.SH
600027.SH
600028.SH
600029.SH
600030.SH
600031.SH
600032.SH
600033.SH
600035.SH
600036.SH
600037.SH
600038.SH
600039.SH
600048.SH
600050.SH
600051.SH
600052.SH
600053.SH
600054.SH
600055.SH
600056.SH
600057.SH
600058.SH
600059.SH
600060.SH
600061.SH
600062.SH
600063.SH
600064.SH
600066.SH
600067.SH
600071.SH
600072.SH
600073.SH
600075.SH
600076.SH
600078.SH
600079.SH
600080.SH
600081.SH
600082.SH
600084.SH
600085.SH
600088.SH
600089.SH
600094.SH
600095.SH
600096.SH
600097.SH
600098.SH
600099.SH
600100.SH
600101.SH
600103.SH
600104.SH
600105.SH
600106.SH
600107.SH
600108.SH
600109.SH
600110.SH
600111.SH
600113.SH
600114.SH
600115.SH
600116.SH
600117.SH
600118.SH
600119.SH
600120.SH
600121.SH
600123.SH
600125.SH
600126.SH
600127.SH
600128.SH
600129.SH
600130.SH
600131.SH
600132.SH
600133.SH
600135.SH
600136.SH
600137.SH
600138.SH
600141.SH
600143.SH
600148.SH
600149.SH
600150.SH
600151.SH
600152.SH
600153.SH
600155.SH
600156.SH
600157.SH
600158.SH
600159.SH
600160.SH
600161.SH
600162.SH
600163.SH
600165.SH
600166.SH
600167.SH
600168.SH
600169.SH
600170.SH
600171.SH
600172.SH
600173.SH
600176.SH
600177.SH
600178.SH
600179.SH
600180.SH
600182.SH
600183.SH
600184.SH
600185.SH
600186.SH
600187.SH
600188.SH
600189.SH
600191.SH
600192.SH
600193.SH
600195.SH
600196.SH
600197.SH
600198.SH
600199.SH
600201.SH
600202.SH
600203.SH
600206.SH
600207.SH
600208.SH
600210.SH
600211.SH
600212.SH
600215.SH
600216.SH
600217.SH
600218.SH
600219.SH
600221.SH
600222.SH
600223.SH
600226.SH
600227.SH
600228.SH
600229.SH
600230.SH
600231.SH
600232.SH
600233.SH
600234.SH
600235.SH
600236.SH
600237.SH
600238.SH
600239.SH
600241.SH
600243.SH
600246.SH
600248.SH
600249.SH
600250.SH
600251.SH
600252.SH
600255.SH
600256.SH
600257.SH
600258.SH
600259.SH
600261.SH
600262.SH
600265.SH
600266.SH
600267.SH
600268.SH
600269.SH
600271.SH
600272.SH
600273.SH
600276.SH
600278.SH
600279.SH
600280.SH
600281.SH
600282.SH
600283.SH
600284.SH
600285.SH
600287.SH
600288.SH
600289.SH
600292.SH
600293.SH
600295.SH
600298.SH
600299.SH
600300.SH
600301.SH
600302.SH
600303.SH
600305.SH
600307.SH
600308.SH
600309.SH
600310.SH
600312.SH
600313.SH
600315.SH
600316.SH
600318.SH
600319.SH
600320.SH
600322.SH
600323.SH
600325.SH
600326.SH
600327.SH
600328.SH
600329.SH
600330.SH
600331.SH
600332.SH
600333.SH
600335.SH
600336.SH
600337.SH
600338.SH
600339.SH
600340.SH
600343.SH
600345.SH
600346.SH
600348.SH
600350.SH
600351.SH
600352.SH
600353.SH
600354.SH
600356.SH
600358.SH
600359.SH
600360.SH
600361.SH
600362.SH
600363.SH
600365.SH
600366.SH
600367.SH
600368.SH
600369.SH
600370.SH
600371.SH
600372.SH
600373.SH
600375.SH
600376.SH
600377.SH
600378.SH
600379.SH
600380.SH
600381.SH
600382.SH
600383.SH
600386.SH
600388.SH
600389.SH
600390.SH
600391.SH
600392.SH
600395.SH
600396.SH
600397.SH
600398.SH
600399.SH
600400.SH
600403.SH
600405.SH
600406.SH
600408.SH
600409.SH
600410.SH
600415.SH
600416.SH
600418.SH
600419.SH
600420.SH
600421.SH
600422.SH
600423.SH
600425.SH
600426.SH
600428.SH
600429.SH
600433.SH
600435.SH
600436.SH
600438.SH
600439.SH
600444.SH
600446.SH
600448.SH
600449.SH
600452.SH
600455.SH
600456.SH
600458.SH
600459.SH
600460.SH
600461.SH
600463.SH
600467.SH
600468.SH
600469.SH
600470.SH
600475.SH
600476.SH
600477.SH
600478.SH
600479.SH
600480.SH
600481.SH
600482.SH
600483.SH
600486.SH
600487.SH
600488.SH
600489.SH
600490.SH
600491.SH
600493.SH
600495.SH
600496.SH
600497.SH
600498.SH
600499.SH
600500.SH
600501.SH
600502.SH
600503.SH
600505.SH
600506.SH
600507.SH
600508.SH
600509.SH
600510.SH
600511.SH
600512.SH
600513.SH
600515.SH
600516.SH
600517.SH
600518.SH
600519.SH
600520.SH
600521.SH
600522.SH
600523.SH
600525.SH
600526.SH
600527.SH
600528.SH
600529.SH
600530.SH
600531.SH
600533.SH
600535.SH
600536.SH
600537.SH
600538.SH
600539.SH
600540.SH
600543.SH
600545.SH
600546.SH
600547.SH
600548.SH
600549.SH
600550.SH
600551.SH
600552.SH
600556.SH
600557.SH
600558.SH
600559.SH
600560.SH
600561.SH
600562.SH
600563.SH
600566.SH
600567.SH
600568.SH
600569.SH
600570.SH
600571.SH
600572.SH
600573.SH
600575.SH
600576.SH
600577.SH
600578.SH
600579.SH
600580.SH
600581.SH
600582.SH
600583.SH
600584.SH
600585.SH
600586.SH
600587.SH
600588.SH
600589.SH
600590.SH
600592.SH
600593.SH
600594.SH
600595.SH
600596.SH
600597.SH
600598.SH
600599.SH
600600.SH
600601.SH
600602.SH
600603.SH
600604.SH
600605.SH
600606.SH
600608.SH
600609.SH
600610.SH
600611.SH
600612.SH
600613.SH
600615.SH
600616.SH
600617.SH
600618.SH
600619.SH
600620.SH
600621.SH
600622.SH
600623.SH
600624.SH
600626.SH
600628.SH
600629.SH
600630.SH
600633.SH
600635.SH
600636.SH
600637.SH
600638.SH
600639.SH
600640.SH
600641.SH
600642.SH
600643.SH
600644.SH
600645.SH
600648.SH
600649.SH
600650.SH
600651.SH
600653.SH
600654.SH
600655.SH
600657.SH
600658.SH
600660.SH
600661.SH
600662.SH
600663.SH
600664.SH
600665.SH
600666.SH
600667.SH
600668.SH
600671.SH
600673.SH
600674.SH
600675.SH
600676.SH
600678.SH
600679.SH
600681.SH
600682.SH
600683.SH
600684.SH
600685.SH
600686.SH
600688.SH
600689.SH
600690.SH
600691.SH
600692.SH
600693.SH
600694.SH
600696.SH
600697.SH
600698.SH
600699.SH
600702.SH
600703.SH
600704.SH
600706.SH
600707.SH
600708.SH
600710.SH
600711.SH
600712.SH
600713.SH
600714.SH
600715.SH
600716.SH
600717.SH
600718.SH
600719.SH
600720.SH
600721.SH
600722.SH
600724.SH
600725.SH
600726.SH
600727.SH
600728.SH
600729.SH
600730.SH
600731.SH
600732.SH
600733.SH
600734.SH
600735.SH
600736.SH
600737.SH
600738.SH
600739.SH
600740.SH
600741.SH
600742.SH
600743.SH
600744.SH
600745.SH
600746.SH
600748.SH
600749.SH
600750.SH
600751.SH
600753.SH
600754.SH
600755.SH
600756.SH
600757.SH
600758.SH
600759.SH
600760.SH
600761.SH
600763.SH
600764.SH
600765.SH
600768.SH
600769.SH
600770.SH
600771.SH
600773.SH
600774.SH
600775.SH
600776.SH
600777.SH
600778.SH
600779.SH
600780.SH
600782.SH
600783.SH
600784.SH
600785.SH
600787.SH
600789.SH
600790.SH
600791.SH
600792.SH
600793.SH
600794.SH
600795.SH
600796.SH
600797.SH
600798.SH
600800.SH
600801.SH
600802.SH
600803.SH
600805.SH
600807.SH
600808.SH
600809.SH
600810.SH
600812.SH
600814.SH
600815.SH
600816.SH
600817.SH
600818.SH
600819.SH
600820.SH
600821.SH
600822.SH
600824.SH
600825.SH
600826.SH
600827.SH
600828.SH
600829.SH
600830.SH
600831.SH
600833.SH
600834.SH
600835.SH
600838.SH
600839.SH
600841.SH
600843.SH
600844.SH
600845.SH
600846.SH
600847.SH
600848.SH
600850.SH
600851.SH
600853.SH
600854.SH
600855.SH
600857.SH
600858.SH
600859.SH
600860.SH
600861.SH
600862.SH
600863.SH
600864.SH
600865.SH
600866.SH
600867.SH
600868.SH
600869.SH
600871.SH
600872.SH
600873.SH
600874.SH
600875.SH
600876.SH
600877.SH
600879.SH
600880.SH
600881.SH
600882.SH
600883.SH
600884.SH
600885.SH
600886.SH
600887.SH
600888.SH
600889.SH
600892.SH
600893.SH
600894.SH
600895.SH
600897.SH
600900.SH
600901.SH
600903.SH
600905.SH
600906.SH
600908.SH
600909.SH
600916.SH
600917.SH
600918.SH
600919.SH
600925.SH
600926.SH
600927.SH
600928.SH
600929.SH
600930.SH
600933.SH
600935.SH
600936.SH
600938.SH
600939.SH
600941.SH
600955.SH
600956.SH
600958.SH
600959.SH
600960.SH
600961.SH
600962.SH
600963.SH
600965.SH
600966.SH
600967.SH
600968.SH
600969.SH
600970.SH
600971.SH
600973.SH
600975.SH
600976.SH
600977.SH
600979.SH
600980.SH
600981.SH
600982.SH
600983.SH
600984.SH
600985.SH
600986.SH
600987.SH
600988.SH
600989.SH
600990.SH
600992.SH
600993.SH
600995.SH
600996.SH
600997.SH
600998.SH
600999.SH
601000.SH
601001.SH
601002.SH
601003.SH
601005.SH
601006.SH
601007.SH
601008.SH
601009.SH
601010.SH
601011.SH
601012.SH
601015.SH
601016.SH
601018.SH
601019.SH
601020.SH
601021.SH
601022.SH
601026.SH
601033.SH
601038.SH
601058.SH
601059.SH
601061.SH
601065.SH
601066.SH
601068.SH
601069.SH
601077.SH
601083.SH
601086.SH
601088.SH
601089.SH
601096.SH
601098.SH
601099.SH
601100.SH
601101.SH
601106.SH
601107.SH
601108.SH
601111.SH
601112.SH
601113.SH
601116.SH
601117.SH
601118.SH
601121.SH
601123.SH
601126.SH
601127.SH
601128.SH
601133.SH
601136.SH
601137.SH
601138.SH
601139.SH
601155.SH
601156.SH
601158.SH
601162.SH
601163.SH
601166.SH
601168.SH
601169.SH
601177.SH
601179.SH
601186.SH
601187.SH
601188.SH
601198.SH
601199.SH
601200.SH
601208.SH
601211.SH
601212.SH
601216.SH
601218.SH
601222.SH
601225.SH
601226.SH
601228.SH
601229.SH
601231.SH
601233.SH
601236.SH
601238.SH
601279.SH
601288.SH
601298.SH
601311.SH
601318.SH
601319.SH
601326.SH
601328.SH
601330.SH
601333.SH
601336.SH
601339.SH
601360.SH
601366.SH
601368.SH
601369.SH
601375.SH
601377.SH
601388.SH
601390.SH
601398.SH
601399.SH
601456.SH
601500.SH
601512.SH
601515.SH
601518.SH
601519.SH
601528.SH
601555.SH
601566.SH
601567.SH
601568.SH
601577.SH
601579.SH
601588.SH
601595.SH
601598.SH
601599.SH
601600.SH
601601.SH
601606.SH
601607.SH
601608.SH
601609.SH
601611.SH
601615.SH
601616.SH
601618.SH
601619.SH
601628.SH
601633.SH
601636.SH
601658.SH
601665.SH
601666.SH
601668.SH
601669.SH
601677.SH
601678.SH
601686.SH
601688.SH
601689.SH
601696.SH
601698.SH
601699.SH
601700.SH
601702.SH
601717.SH
601718.SH
601727.SH
601728.SH
601766.SH
601777.SH
601778.SH
601788.SH
601789.SH
601798.SH
601799.SH
601800.SH
601801.SH
601808.SH
601811.SH
601816.SH
601818.SH
601825.SH
601827.SH
601828.SH
601838.SH
601857.SH
601858.SH
601860.SH
601865.SH
601866.SH
601868.SH
601869.SH
601872.SH
601877.SH
601878.SH
601880.SH
601881.SH
601882.SH
601886.SH
601888.SH
601890.SH
601898.SH
601899.SH
601900.SH
601901.SH
601908.SH
601916.SH
601918.SH
601919.SH
601921.SH
601928.SH
601929.SH
601933.SH
601939.SH
601949.SH
601952.SH
601956.SH
601958.SH
601963.SH
601965.SH
601966.SH
601968.SH
601969.SH
601975.SH
601985.SH
601988.SH
601990.SH
601991.SH
601992.SH
601995.SH
601996.SH
601997.SH
601998.SH
601999.SH
603000.SH
603001.SH
603002.SH
603004.SH
603005.SH
603006.SH
603007.SH
603008.SH
603009.SH
603010.SH
603011.SH
603012.SH
603013.SH
603014.SH
603015.SH
603016.SH
603017.SH
603018.SH
603019.SH
603020.SH
603021.SH
603022.SH
603023.SH
603025.SH
603026.SH
603027.SH
603028.SH
603029.SH
603030.SH
603031.SH
603032.SH
603033.SH
603035.SH
603036.SH
603037.SH
603038.SH
603039.SH
603040.SH
603041.SH
603042.SH
603043.SH
603045.SH
603048.SH
603049.SH
603050.SH
603051.SH
603052.SH
603053.SH
603055.SH
603057.SH
603058.SH
603059.SH
603060.SH
603061.SH
603062.SH
603063.SH
603065.SH
603066.SH
603067.SH
603068.SH
603069.SH
603070.SH
603071.SH
603072.SH
603073.SH
603075.SH
603076.SH
603077.SH
603078.SH
603079.SH
603080.SH
603081.SH
603082.SH
603083.SH
603085.SH
603086.SH
603087.SH
603088.SH
603089.SH
603090.SH
603091.SH
603092.SH
603093.SH
603095.SH
603096.SH
603097.SH
603098.SH
603099.SH
603100.SH
603101.SH
603102.SH
603103.SH
603105.SH
603106.SH
603107.SH
603108.SH
603109.SH
603110.SH
603111.SH
603112.SH
603113.SH
603115.SH
603116.SH
603117.SH
603118.SH
603119.SH
603120.SH
603121.SH
603122.SH
603123.SH
603124.SH
603125.SH
603126.SH
603127.SH
603128.SH
603129.SH
603130.SH
603131.SH
603132.SH
603135.SH
603136.SH
603137.SH
603138.SH
603139.SH
603150.SH
603151.SH
603153.SH
603155.SH
603156.SH
603158.SH
603159.SH
603160.SH
603161.SH
603162.SH
603163.SH
603165.SH
603166.SH
603167.SH
603168.SH
603169.SH
603170.SH
603171.SH
603172.SH
603173.SH
603175.SH
603176.SH
603177.SH
603178.SH
603179.SH
603180.SH
603181.SH
603182.SH
603183.SH
603185.SH
603186.SH
603187.SH
603188.SH
603189.SH
603190.SH
603191.SH
603192.SH
603193.SH
603194.SH
603195.SH
603196.SH
603197.SH
603198.SH
603199.SH
603200.SH
603201.SH
603202.SH
603203.SH
603205.SH
603206.SH
603207.SH
603208.SH
603209.SH
603210.SH
603211.SH
603212.SH
603213.SH
603214.SH
603215.SH
603216.SH
603217.SH
603218.SH
603219.SH
603220.SH
603221.SH
603222.SH
603223.SH
603225.SH
603226.SH
603227.SH
603228.SH
603229.SH
603230.SH
603231.SH
603232.SH
603233.SH
603235.SH
603236.SH
603237.SH
603238.SH
603239.SH
603248.SH
603255.SH
603256.SH
603257.SH
603258.SH
603259.SH
603260.SH
603261.SH
603262.SH
603266.SH
603267.SH
603268.SH
603269.SH
603270.SH
603271.SH
603272.SH
603273.SH
603275.SH
603276.SH
603277.SH
603278.SH
603279.SH
603280.SH
603281.SH
603282.SH
603283.SH
603284.SH
603285.SH
603286.SH
603288.SH
603289.SH
603290.SH
603291.SH
603293.SH
603296.SH
603297.SH
603298.SH
603299.SH
603300.SH
603301.SH
603303.SH
603305.SH
603306.SH
603307.SH
603308.SH
603309.SH
603310.SH
603311.SH
603312.SH
603313.SH
603315.SH
603316.SH
603317.SH
603318.SH
603319.SH
603320.SH
603321.SH
603322.SH
603323.SH
603324.SH
603325.SH
603326.SH
603327.SH
603328.SH
603329.SH
603330.SH
603331.SH
603332.SH
603333.SH
603334.SH
603335.SH
603336.SH
603337.SH
603338.SH
603339.SH
603341.SH
603344.SH
603345.SH
603348.SH
603350.SH
603351.SH
603352.SH
603353.SH
603355.SH
603356.SH
603357.SH
603358.SH
603359.SH
603360.SH
603363.SH
603365.SH
603366.SH
603367.SH
603368.SH
603369.SH
603370.SH
603373.SH
603375.SH
603376.SH
603377.SH
603378.SH
603379.SH
603380.SH
603381.SH
603382.SH
603383.SH
603385.SH
603386.SH
603387.SH
603389.SH
603390.SH
603391.SH
603392.SH
603393.SH
603395.SH
603396.SH
603398.SH
603399.SH
603400.SH
603402.SH
603406.SH
603407.SH
603408.SH
603409.SH
603416.SH
603418.SH
603421.SH
603429.SH
603435.SH
603439.SH
603444.SH
603448.SH
603456.SH
603458.SH
603459.SH
603466.SH
603468.SH
603477.SH
603486.SH
603488.SH
603489.SH
603496.SH
603499.SH
603500.SH
603501.SH
603505.SH
603506.SH
603507.SH
603508.SH
603511.SH
603515.SH
603516.SH
603517.SH
603518.SH
603519.SH
603520.SH
603527.SH
603528.SH
603529.SH
603530.SH
603533.SH
603535.SH
603536.SH
603538.SH
603551.SH
603556.SH
603557.SH
603558.SH
603559.SH
603565.SH
603566.SH
603567.SH
603568.SH
603569.SH
603577.SH
603578.SH
603579.SH
603580.SH
603583.SH
603585.SH
603586.SH
603587.SH
603588.SH
603589.SH
603590.SH
603595.SH
603596.SH
603598.SH
603599.SH
603600.SH
603601.SH
603602.SH
603605.SH
603606.SH
603607.SH
603608.SH
603609.SH
603610.SH
603611.SH
603612.SH
603613.SH
603615.SH
603616.SH
603617.SH
603618.SH
603619.SH
603626.SH
603628.SH
603629.SH
603630.SH
603633.SH
603636.SH
603637.SH
603638.SH
603639.SH
603648.SH
603650.SH
603655.SH
603656.SH
603657.SH
603658.SH
603659.SH
603660.SH
603661.SH
603662.SH
603663.SH
603665.SH
603666.SH
603667.SH
603668.SH
603669.SH
603676.SH
603677.SH
603678.SH
603679.SH
603680.SH
603681.SH
603682.SH
603683.SH
603685.SH
603686.SH
603687.SH
603688.SH
603689.SH
603690.SH
603693.SH
603696.SH
603697.SH
603698.SH
603699.SH
603700.SH
603701.SH
603703.SH
603706.SH
603707.SH
603708.SH
603709.SH
603711.SH
603712.SH
603713.SH
603716.SH
603717.SH
603718.SH
603719.SH
603721.SH
603722.SH
603725.SH
603726.SH
603727.SH
603728.SH
603729.SH
603730.SH
603733.SH
603737.SH
603738.SH
603739.SH
603755.SH
603757.SH
603758.SH
603759.SH
603766.SH
603767.SH
603768.SH
603773.SH
603776.SH
603777.SH
603778.SH
603779.SH
603786.SH
603787.SH
603788.SH
603789.SH
603790.SH
603797.SH
603798.SH
603799.SH
603800.SH
603801.SH
603803.SH
603806.SH
603808.SH
603809.SH
603810.SH
603811.SH
603813.SH
603815.SH
603816.SH
603817.SH
603818.SH
603819.SH
603822.SH
603823.SH
603825.SH
603826.SH
603828.SH
603829.SH
603833.SH
603836.SH
603838.SH
603839.SH
603843.SH
603848.SH
603855.SH
603856.SH
603858.SH
603859.SH
603860.SH
603861.SH
603863.SH
603866.SH
603867.SH
603868.SH
603869.SH
603871.SH
603876.SH
603877.SH
603878.SH
603879.SH
603880.SH
603881.SH
603882.SH
603883.SH
603885.SH
603886.SH
603887.SH
603888.SH
603889.SH
603890.SH
603893.SH
603895.SH
603896.SH
603897.SH
603898.SH
603899.SH
603900.SH
603901.SH
603903.SH
603906.SH
603908.SH
603909.SH
603912.SH
603915.SH
603916.SH
603917.SH
603918.SH
603919.SH
603920.SH
603922.SH
603926.SH
603927.SH
603928.SH
603929.SH
603931.SH
603933.SH
603936.SH
603937.SH
603938.SH
603939.SH
603948.SH
603949.SH
603950.SH
603955.SH
603956.SH
603958.SH
603959.SH
603960.SH
603966.SH
603967.SH
603968.SH
603969.SH
603970.SH
603976.SH
603977.SH
603978.SH
603979.SH
603980.SH
603982.SH
603983.SH
603985.SH
603986.SH
603987.SH
603988.SH
603989.SH
603990.SH
603991.SH
603992.SH
603993.SH
603995.SH
603997.SH
603998.SH
603999.SH
605001.SH
605003.SH
605005.SH
605006.SH
605007.SH
605008.SH
605009.SH
605011.SH
605016.SH
605018.SH
605020.SH
605028.SH
605033.SH
605050.SH
605055.SH
605056.SH
605058.SH
605060.SH
605066.SH
605068.SH
605069.SH
605077.SH
605080.SH
605081.SH
605086.SH
605088.SH
605089.SH
605090.SH
605098.SH
605099.SH
605100.SH
605108.SH
605111.SH
605116.SH
605117.SH
605118.SH
605122.SH
605123.SH
605128.SH
605133.SH
605136.SH
605138.SH
605151.SH
605155.SH
605158.SH
605162.SH
605166.SH
605167.SH
605168.SH
605169.SH
605177.SH
605178.SH
605179.SH
605180.SH
605183.SH
605186.SH
605188.SH
605189.SH
605196.SH
605198.SH
605199.SH
605208.SH
605218.SH
605222.SH
605228.SH
605255.SH
605258.SH
605259.SH
605266.SH
605268.SH
605277.SH
605286.SH
605287.SH
605288.SH
605289.SH
605296.SH
605298.SH
605299.SH
605300.SH
605303.SH
605305.SH
605318.SH
605319.SH
605333.SH
605336.SH
605337.SH
605338.SH
605339.SH
605358.SH
605365.SH
605366.SH
605368.SH
605369.SH
605376.SH
605377.SH
605378.SH
605388.SH
605389.SH
605398.SH
605399.SH
605488.SH
605499.SH
605500.SH
605507.SH
605555.SH
605566.SH
605567.SH
605577.SH
605580.SH
605588.SH
605589.SH
605598.SH
605599.SH
688001.SH
688002.SH
688003.SH
688004.SH
688005.SH
688006.SH
688007.SH
688008.SH
688009.SH
688010.SH
688011.SH
688012.SH
688013.SH
688015.SH
688016.SH
688017.SH
688018.SH
688019.SH
688020.SH
688021.SH
688022.SH
688023.SH
688025.SH
688026.SH
688027.SH
688028.SH
688029.SH
688030.SH
688031.SH
688032.SH
688033.SH
688035.SH
688036.SH
688037.SH
688038.SH
688039.SH
688041.SH
688045.SH
688046.SH
688047.SH
688048.SH
688049.SH
688050.SH
688051.SH
688052.SH
688053.SH
688055.SH
688056.SH
688057.SH
688058.SH
688059.SH
688060.SH
688061.SH
688062.SH
688063.SH
688065.SH
688066.SH
688067.SH
688068.SH
688069.SH
688070.SH
688071.SH
688072.SH
688073.SH
688075.SH
688076.SH
688077.SH
688078.SH
688079.SH
688080.SH
688081.SH
688082.SH
688083.SH
688084.SH
688085.SH
688087.SH
688088.SH
688089.SH
688090.SH
688091.SH
688092.SH
688093.SH
688095.SH
688096.SH
688097.SH
688098.SH
688099.SH
688100.SH
688101.SH
688102.SH
688103.SH
688105.SH
688106.SH
688107.SH
688108.SH
688109.SH
688110.SH
688111.SH
688112.SH
688113.SH
688114.SH
688115.SH
688116.SH
688117.SH
688118.SH
688119.SH
688120.SH
688121.SH
688122.SH
688123.SH
688125.SH
688126.SH
688127.SH
688128.SH
688129.SH
688130.SH
688131.SH
688132.SH
688133.SH
688135.SH
688136.SH
688137.SH
688138.SH
688139.SH
688141.SH
688143.SH
688146.SH
688147.SH
688148.SH
688150.SH
688151.SH
688152.SH
688153.SH
688155.SH
688156.SH
688157.SH
688158.SH
688159.SH
688160.SH
688161.SH
688162.SH
688163.SH
688165.SH
688166.SH
688167.SH
688168.SH
688169.SH
688170.SH
688171.SH
688172.SH
688173.SH
688175.SH
688176.SH
688177.SH
688178.SH
688179.SH
688180.SH
688181.SH
688182.SH
688183.SH
688184.SH
688185.SH
688186.SH
688187.SH
688188.SH
688189.SH
688190.SH
688191.SH
688192.SH
688193.SH
688195.SH
688196.SH
688197.SH
688198.SH
688199.SH
688200.SH
688201.SH
688202.SH
688203.SH
688205.SH
688206.SH
688207.SH
688208.SH
688209.SH
688210.SH
688211.SH
688212.SH
688213.SH
688215.SH
688216.SH
688217.SH
688218.SH
688219.SH
688220.SH
688221.SH
688222.SH
688223.SH
688225.SH
688226.SH
688227.SH
688228.SH
688229.SH
688230.SH
688231.SH
688232.SH
688233.SH
688234.SH
688235.SH
688236.SH
688237.SH
688238.SH
688239.SH
688244.SH
688246.SH
688247.SH
688248.SH
688249.SH
688251.SH
688252.SH
688253.SH
688255.SH
688256.SH
688257.SH
688258.SH
688259.SH
688260.SH
688261.SH
688262.SH
688265.SH
688266.SH
688267.SH
688268.SH
688269.SH
688270.SH
688271.SH
688272.SH
688273.SH
688275.SH
688276.SH
688277.SH
688278.SH
688279.SH
688280.SH
688281.SH
688282.SH
688283.SH
688285.SH
688286.SH
688287.SH
688288.SH
688289.SH
688290.SH
688291.SH
688292.SH
688293.SH
688295.SH
688296.SH
688297.SH
688298.SH
688299.SH
688300.SH
688301.SH
688302.SH
688303.SH
688305.SH
688306.SH
688307.SH
688308.SH
688309.SH
688310.SH
688311.SH
688312.SH
688313.SH
688314.SH
688315.SH
688316.SH
688317.SH
688318.SH
688319.SH
688320.SH
688321.SH
688322.SH
688323.SH
688325.SH
688326.SH
688327.SH
688328.SH
688329.SH
688330.SH
688331.SH
688332.SH
688333.SH
688334.SH
688335.SH
688336.SH
688337.SH
688338.SH
688339.SH
688343.SH
688345.SH
688347.SH
688348.SH
688349.SH
688350.SH
688351.SH
688352.SH
688353.SH
688355.SH
688356.SH
688357.SH
688358.SH
688359.SH
688360.SH
688361.SH
688362.SH
688363.SH
688365.SH
688366.SH
688367.SH
688368.SH
688369.SH
688370.SH
688371.SH
688372.SH
688373.SH
688375.SH
688376.SH
688377.SH
688378.SH
688379.SH
688380.SH
688381.SH
688382.SH
688383.SH
688385.SH
688386.SH
688387.SH
688388.SH
688389.SH
688390.SH
688391.SH
688392.SH
688393.SH
688395.SH
688396.SH
688398.SH
688399.SH
688400.SH
688401.SH
688403.SH
688408.SH
688409.SH
688410.SH
688411.SH
688416.SH
688418.SH
688419.SH
688420.SH
688425.SH
688426.SH
688428.SH
688429.SH
688432.SH
688433.SH
688435.SH
688439.SH
688443.SH
688448.SH
688449.SH
688450.SH
688455.SH
688456.SH
688458.SH
688459.SH
688466.SH
688468.SH
688469.SH
688472.SH
688475.SH
688478.SH
688479.SH
688480.SH
688484.SH
688485.SH
688486.SH
688488.SH
688489.SH
688496.SH
688498.SH
688499.SH
688500.SH
688501.SH
688502.SH
688503.SH
688505.SH
688506.SH
688507.SH
688508.SH
688509.SH
688510.SH
688511.SH
688512.SH
688513.SH
688515.SH
688516.SH
688517.SH
688518.SH
688519.SH
688520.SH
688521.SH
688522.SH
688523.SH
688525.SH
688526.SH
688528.SH
688529.SH
688530.SH
688531.SH
688533.SH
688535.SH
688536.SH
688538.SH
688539.SH
688543.SH
688545.SH
688548.SH
688549.SH
688550.SH
688551.SH
688552.SH
688553.SH
688556.SH
688557.SH
688558.SH
688559.SH
688560.SH
688561.SH
688562.SH
688563.SH
688565.SH
688566.SH
688567.SH
688568.SH
688569.SH
688570.SH
688571.SH
688573.SH
688575.SH
688576.SH
688577.SH
688578.SH
688579.SH
688580.SH
688581.SH
688582.SH
688583.SH
688584.SH
688585.SH
688586.SH
688588.SH
688589.SH
688590.SH
688591.SH
688592.SH
688593.SH
688595.SH
688596.SH
688597.SH
688598.SH
688599.SH
688600.SH
688601.SH
688602.SH
688603.SH
688605.SH
688606.SH
688607.SH
688608.SH
688609.SH
688610.SH
688611.SH
688612.SH
688613.SH
688615.SH
688616.SH
688617.SH
688618.SH
688619.SH
688620.SH
688621.SH
688622.SH
688623.SH
688625.SH
688626.SH
688627.SH
688628.SH
688629.SH
688630.SH
688631.SH
688633.SH
688635.SH
688636.SH
688638.SH
688639.SH
688646.SH
688648.SH
688651.SH
688652.SH
688653.SH
688655.SH
688656.SH
688657.SH
688658.SH
688659.SH
688660.SH
688661.SH
688662.SH
688663.SH
688665.SH
688667.SH
688668.SH
688669.SH
688670.SH
688671.SH
688676.SH
688677.SH
688678.SH
688679.SH
688680.SH
688681.SH
688682.SH
688683.SH
688685.SH
688686.SH
688687.SH
688689.SH
688690.SH
688691.SH
688692.SH
688693.SH
688695.SH
688696.SH
688697.SH
688698.SH
688699.SH
688700.SH
688701.SH
688702.SH
688707.SH
688708.SH
688709.SH
688710.SH
688711.SH
688712.SH
688716.SH
688717.SH
688718.SH
688719.SH
688720.SH
688721.SH
688722.SH
688726.SH
688727.SH
688728.SH
688729.SH
688733.SH
688737.SH
688739.SH
688750.SH
688755.SH
688757.SH
688758.SH
688759.SH
688765.SH
688766.SH
688767.SH
688768.SH
688772.SH
688775.SH
688776.SH
688777.SH
688778.SH
688779.SH
688781.SH
688783.SH
688785.SH
688786.SH
688787.SH
688788.SH
688789.SH
688790.SH
688793.SH
688795.SH
688796.SH
688797.SH
688798.SH
688799.SH
688800.SH
688801.SH
688802.SH
688805.SH
688806.SH
688807.SH
688808.SH
688809.SH
688811.SH
688813.SH
688816.SH
688818.SH
688819.SH
688820.SH
688825.SH
688826.SH
688828.SH
688835.SH
688836.SH
688981.SH
689009.SH
830779.BJ
830799.BJ
830809.BJ
830832.BJ
830839.BJ
830879.BJ
830896.BJ
830946.BJ
830964.BJ
830974.BJ
831010.BJ
831039.BJ
831087.BJ
831152.BJ
831167.BJ
831175.BJ
831195.BJ
831278.BJ
831304.BJ
831305.BJ
831370.BJ
831396.BJ
831445.BJ
831526.BJ
831627.BJ
831641.BJ
831689.BJ
831726.BJ
831768.BJ
831832.BJ
831834.BJ
831855.BJ
831856.BJ
831906.BJ
831961.BJ
832000.BJ
832023.BJ
832089.BJ
832110.BJ
832145.BJ
832149.BJ
832171.BJ
832175.BJ
832225.BJ
832278.BJ
832419.BJ
832469.BJ
832471.BJ
832491.BJ
832522.BJ
832566.BJ
832651.BJ
832662.BJ
832735.BJ
832786.BJ
832802.BJ
832876.BJ
832885.BJ
832978.BJ
832982.BJ
833030.BJ
833075.BJ
833171.BJ
833230.BJ
833266.BJ
833284.BJ
833346.BJ
833394.BJ
833427.BJ
833429.BJ
833454.BJ
833455.BJ
833509.BJ
833523.BJ
833533.BJ
833575.BJ
833580.BJ
833751.BJ
833781.BJ
833819.BJ
833873.BJ
833914.BJ
833943.BJ
834014.BJ
834021.BJ
834033.BJ
834058.BJ
834062.BJ
834261.BJ
834407.BJ
834415.BJ
834475.BJ
834599.BJ
834639.BJ
834682.BJ
834765.BJ
834770.BJ
834950.BJ
835174.BJ
835179.BJ
835184.BJ
835185.BJ
835207.BJ
835237.BJ
835305.BJ
835368.BJ
835438.BJ
835508.BJ
835579.BJ
835640.BJ
835670.BJ
835857.BJ
835892.BJ
835985.BJ
836077.BJ
836149.BJ
836208.BJ
836221.BJ
836239.BJ
836247.BJ
836260.BJ
836263.BJ
836270.BJ
836395.BJ
836414.BJ
836419.BJ
836422.BJ
836433.BJ
836504.BJ
836547.BJ
836675.BJ
836699.BJ
836717.BJ
836720.BJ
836807.BJ
836826.BJ
836871.BJ
836892.BJ
836942.BJ
836957.BJ
836961.BJ
837006.BJ
837023.BJ
837046.BJ
837092.BJ
837174.BJ
837212.BJ
837242.BJ
837344.BJ
837403.BJ
837592.BJ
837663.BJ
837748.BJ
837821.BJ
838030.BJ
838163.BJ
838171.BJ
838227.BJ
838262.BJ
838275.BJ
838402.BJ
838670.BJ
838701.BJ
838810.BJ
838837.BJ
838924.BJ
838971.BJ
839167.BJ
839273.BJ
839371.BJ
839493.BJ
839680.BJ
839719.BJ
839725.BJ
839729.BJ
839790.BJ
839792.BJ
839946.BJ
870199.BJ
870204.BJ
870299.BJ
870357.BJ
870436.BJ
870508.BJ
870656.BJ
870726.BJ
870866.BJ
870976.BJ
871245.BJ
871263.BJ
871396.BJ
871478.BJ
871553.BJ
871634.BJ
871642.BJ
871694.BJ
871753.BJ
871857.BJ
871970.BJ
871981.BJ
872190.BJ
872351.BJ
872374.BJ
872392.BJ
872541.BJ
872808.BJ
872895.BJ
872925.BJ
872931.BJ
872953.BJ
873001.BJ
873122.BJ
873132.BJ
873152.BJ
873167.BJ
873169.BJ
873223.BJ
873305.BJ
873339.BJ
873527.BJ
873570.BJ
873576.BJ
873593.BJ
873665.BJ
873679.BJ
873690.BJ
873693.BJ
873703.BJ
873706.BJ
873726.BJ
873806.BJ
873833.BJ
920000.BJ
920001.BJ
920002.BJ
920003.BJ
920005.BJ
920006.BJ
920007.BJ
920008.BJ
920009.BJ
920010.BJ
920011.BJ
920012.BJ
920014.BJ
920015.BJ
920016.BJ
920017.BJ
920018.BJ
920019.BJ
920020.BJ
920021.BJ
920022.BJ
920023.BJ
920026.BJ
920027.BJ
920028.BJ
920029.BJ
920030.BJ
920033.BJ
920035.BJ
920036.BJ
920037.BJ
920038.BJ
920039.BJ
920045.BJ
920046.BJ
920047.BJ
920050.BJ
920055.BJ
920056.BJ
920057.BJ
920058.BJ
920059.BJ
920060.BJ
920061.BJ
920062.BJ
920065.BJ
920066.BJ
920068.BJ
920069.BJ
920071.BJ
920072.BJ
920075.BJ
920076.BJ
920077.BJ
920078.BJ
920079.BJ
920080.BJ
920081.BJ
920082.BJ
920083.BJ
920086.BJ
920087.BJ
920088.BJ
920089.BJ
920090.BJ
920091.BJ
920092.BJ
920093.BJ
920096.BJ
920098.BJ
920099.BJ
920100.BJ
920101.BJ
920106.BJ
920107.BJ
920108.BJ
920110.BJ
920111.BJ
920112.BJ
920116.BJ
920117.BJ
920118.BJ
920119.BJ
920121.BJ
920122.BJ
920123.BJ
920124.BJ
920125.BJ
920126.BJ
920128.BJ
920130.BJ
920132.BJ
920136.BJ
920138.BJ
920139.BJ
920145.BJ
920146.BJ
920149.BJ
920152.BJ
920156.BJ
920158.BJ
920159.BJ
920160.BJ
920161.BJ
920163.BJ
920165.BJ
920166.BJ
920167.BJ
920168.BJ
920169.BJ
920171.BJ
920174.BJ
920175.BJ
920176.BJ
920177.BJ
920178.BJ
920179.BJ
920180.BJ
920181.BJ
920183.BJ
920184.BJ
920185.BJ
920186.BJ
920187.BJ
920188.BJ
920189.BJ
920190.BJ
920191.BJ
920193.BJ
920195.BJ
920198.BJ
920199.BJ
920200.BJ
920204.BJ
920206.BJ
920207.BJ
920208.BJ
920211.BJ
920212.BJ
920218.BJ
920220.BJ
920221.BJ
920222.BJ
920223.BJ
920225.BJ
920227.BJ
920230.BJ
920237.BJ
920238.BJ
920239.BJ
920242.BJ
920245.BJ
920247.BJ
920249.BJ
920252.BJ
920258.BJ
920260.BJ
920261.BJ
920262.BJ
920263.BJ
920266.BJ
920267.BJ
920268.BJ
920269.BJ
920270.BJ
920271.BJ
920273.BJ
920274.BJ
920275.BJ
920278.BJ
920284.BJ
920288.BJ
920289.BJ
920299.BJ
920300.BJ
920304.BJ
920305.BJ
920339.BJ
920344.BJ
920346.BJ
920351.BJ
920357.BJ
920363.BJ
920367.BJ
920368.BJ
920370.BJ
920371.BJ
920374.BJ
920375.BJ
920378.BJ
920392.BJ
920394.BJ
920395.BJ
920396.BJ
920402.BJ
920403.BJ
920405.BJ
920407.BJ
920414.BJ
920415.BJ
920418.BJ
920419.BJ
920422.BJ
920425.BJ
920427.BJ
920429.BJ
920433.BJ
920436.BJ
920438.BJ
920445.BJ
920454.BJ
920455.BJ
920469.BJ
920471.BJ
920475.BJ
920476.BJ
920478.BJ
920489.BJ
920491.BJ
920493.BJ
920496.BJ
920504.BJ
920505.BJ
920508.BJ
920509.BJ
920510.BJ
920519.BJ
920522.BJ
920523.BJ
920526.BJ
920527.BJ
920533.BJ
920541.BJ
920547.BJ
920553.BJ
920556.BJ
920564.BJ
920566.BJ
920570.BJ
920571.BJ
920575.BJ
920576.BJ
920578.BJ
920579.BJ
920580.BJ
920592.BJ
920593.BJ
920599.BJ
920608.BJ
920627.BJ
920634.BJ
920639.BJ
920640.BJ
920641.BJ
920642.BJ
920651.BJ
920656.BJ
920662.BJ
920663.BJ
920665.BJ
920670.BJ
920675.BJ
920679.BJ
920682.BJ
920685.BJ
920689.BJ
920690.BJ
920693.BJ
920694.BJ
920699.BJ
920701.BJ
920703.BJ
920706.BJ
920717.BJ
920718.BJ
920719.BJ
920720.BJ
920725.BJ
920726.BJ
920729.BJ
920735.BJ
920748.BJ
920751.BJ
920753.BJ
920765.BJ
920768.BJ
920770.BJ
920779.BJ
920781.BJ
920786.BJ
920790.BJ
920792.BJ
920799.BJ
920802.BJ
920806.BJ
920807.BJ
920808.BJ
920809.BJ
920810.BJ
920819.BJ
920821.BJ
920826.BJ
920832.BJ
920833.BJ
920834.BJ
920837.BJ
920839.BJ
920855.BJ
920856.BJ
920857.BJ
920866.BJ
920870.BJ
920871.BJ
920873.BJ
920876.BJ
920879.BJ
920885.BJ
920892.BJ
920895.BJ
920896.BJ
920906.BJ
920914.BJ
920924.BJ
920925.BJ
920926.BJ
920931.BJ
920932.BJ
920942.BJ
920943.BJ
920946.BJ
920950.BJ
920953.BJ
920957.BJ
920961.BJ
920964.BJ
920970.BJ
920971.BJ
920974.BJ
920976.BJ
920978.BJ
920981.BJ
920982.BJ
920985.BJ
920992.BJ""".split()
NO_PRICE_SYMBOLS = """430017.BJ
430047.BJ
430090.BJ
430139.BJ
430198.BJ
430300.BJ
430418.BJ
430425.BJ
430476.BJ
430478.BJ
430489.BJ
430510.BJ
430556.BJ
430564.BJ
430685.BJ
430718.BJ
830779.BJ
830799.BJ
830809.BJ
830832.BJ
830839.BJ
830879.BJ
830896.BJ
830946.BJ
830964.BJ
830974.BJ
831010.BJ
831039.BJ
831087.BJ
831152.BJ
831167.BJ
831175.BJ
831195.BJ
831278.BJ
831304.BJ
831305.BJ
831370.BJ
831396.BJ
831445.BJ
831526.BJ
831627.BJ
831641.BJ
831689.BJ
831726.BJ
831768.BJ
831832.BJ
831834.BJ
831855.BJ
831856.BJ
831906.BJ
831961.BJ
832000.BJ
832023.BJ
832089.BJ
832110.BJ
832145.BJ
832149.BJ
832171.BJ
832175.BJ
832225.BJ
832278.BJ
832419.BJ
832469.BJ
832471.BJ
832491.BJ
832522.BJ
832566.BJ
832651.BJ
832662.BJ
832735.BJ
832786.BJ
832802.BJ
832876.BJ
832885.BJ
832978.BJ
832982.BJ
833030.BJ
833075.BJ
833171.BJ
833230.BJ
833266.BJ
833284.BJ
833346.BJ
833394.BJ
833427.BJ
833429.BJ
833454.BJ
833455.BJ
833509.BJ
833523.BJ
833533.BJ
833575.BJ
833580.BJ
833751.BJ
833781.BJ
833819.BJ
833873.BJ
833914.BJ
833943.BJ
834014.BJ
834021.BJ
834033.BJ
834058.BJ
834062.BJ
834261.BJ
834407.BJ
834415.BJ
834475.BJ
834599.BJ
834639.BJ
834682.BJ
834765.BJ
834770.BJ
834950.BJ
835174.BJ
835179.BJ
835184.BJ
835185.BJ
835207.BJ
835237.BJ
835305.BJ
835368.BJ
835438.BJ
835508.BJ
835579.BJ
835640.BJ
835670.BJ
835857.BJ
835892.BJ
835985.BJ
836077.BJ
836149.BJ
836208.BJ
836221.BJ
836239.BJ
836247.BJ
836260.BJ
836263.BJ
836270.BJ
836395.BJ
836414.BJ
836419.BJ
836422.BJ
836433.BJ
836504.BJ
836547.BJ
836675.BJ
836699.BJ
836717.BJ
836720.BJ
836807.BJ
836826.BJ
836871.BJ
836892.BJ
836942.BJ
836957.BJ
836961.BJ
837006.BJ
837023.BJ
837046.BJ
837092.BJ
837174.BJ
837212.BJ
837242.BJ
837344.BJ
837403.BJ
837592.BJ
837663.BJ
837748.BJ
837821.BJ
838030.BJ
838163.BJ
838171.BJ
838227.BJ
838262.BJ
838275.BJ
838402.BJ
838670.BJ
838701.BJ
838810.BJ
838837.BJ
838924.BJ
838971.BJ
839167.BJ
839273.BJ
839371.BJ
839493.BJ
839680.BJ
839719.BJ
839725.BJ
839729.BJ
839790.BJ
839792.BJ
839946.BJ
870199.BJ
870204.BJ
870299.BJ
870357.BJ
870436.BJ
870508.BJ
870656.BJ
870726.BJ
870866.BJ
870976.BJ
871245.BJ
871263.BJ
871396.BJ
871478.BJ
871553.BJ
871634.BJ
871642.BJ
871694.BJ
871753.BJ
871857.BJ
871970.BJ
871981.BJ
872190.BJ
872351.BJ
872374.BJ
872392.BJ
872541.BJ
872808.BJ
872895.BJ
872925.BJ
872931.BJ
872953.BJ
873001.BJ
873122.BJ
873132.BJ
873152.BJ
873167.BJ
873169.BJ
873223.BJ
873305.BJ
873339.BJ
873527.BJ
873570.BJ
873576.BJ
873593.BJ
873665.BJ
873679.BJ
873690.BJ
873693.BJ
873703.BJ
873706.BJ
873726.BJ
873806.BJ
873833.BJ""".split()
BATCH_SIZE = 100
ROW_LIMIT = 1000
PERIODS = ["2025q4", "2026q1", "2026q2"]
PART_LIMIT = 45 * 1024 ** 2


def pull_financials():
    required = (query, get_fundamentals, balance, income_sq,
                read_file, write_file, list_file, remove_file)
    tag = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
    prefix = "ths_quarterly_2025q4_2026q2_" + tag
    outputs = []
    audit = []
    calls = [0]

    file_sizes = {}

    def save(frame, label):
        name = prefix + "_" + label + ".pkl.gz"
        temp = prefix + "_temp.pkl.gz"
        try:
            frame.to_pickle(temp, compression="gzip", protocol=4)
            restored = pd.read_pickle(temp, compression="gzip")
            if not restored.equals(frame) or restored.attrs != frame.attrs:
                raise RuntimeError("文件回读不一致：" + label)
            del restored
            payload = read_file(temp)
            if not isinstance(payload, bytes):
                raise TypeError("read_file未返回bytes")
            count = (len(payload) + PART_LIMIT - 1) // PART_LIMIT
            for index in range(count):
                piece = payload[index * PART_LIMIT:(index + 1) * PART_LIMIT]
                target = name if count == 1 else name + ".part{:03d}".format(index + 1)
                written = write_file(target, piece)
                if written != len(piece) or read_file(target) != piece:
                    raise RuntimeError("分卷写入回读失败：" + target)
                if len(piece) > PART_LIMIT:
                    raise RuntimeError("分卷超限")
                if target not in outputs:
                    outputs.append(target)
                file_sizes[target] = len(piece)
                print("已核验：", target, "大小MiB：", round(len(piece) / 1024 ** 2, 3))
            if count > 1:
                print("分卷须按part序号拼接为", name, "后再用pd.read_pickle读取")
        finally:
            if temp in list_file("", isfile=True):
                remove_file(temp, trash=False)
        if temp in list_file("", isfile=True):
            raise RuntimeError("临时文件未清理")

    def field(raw, table_name, suffix):
        candidates = [table_name + "_stat_" + suffix, table_name + "_" + suffix]
        matches = [c for c in candidates if c in raw.columns]
        if len(matches) != 1:
            raise RuntimeError("无法唯一确认字段 " + suffix + "；实际字段：" + str(raw.columns.tolist()))
        return matches[0]

    # 先验证小文件，避免拉完才发现当前内核不允许写文件。
    probe_name = prefix + "_probe.pkl.gz"
    try:
        probe = pd.DataFrame({"test": [1]})
        probe.to_pickle(probe_name, compression="gzip", protocol=4)
        if not pd.read_pickle(probe_name).equals(probe):
            raise RuntimeError("压缩回读预检失败")
    finally:
        if probe_name in list_file("", isfile=True):
            remove_file(probe_name, trash=False)
    if probe_name in list_file("", isfile=True):
        raise RuntimeError("预检文件清理失败")

    def fetch(table, table_name, period, symbols):
        calls[0] += 1
        raw = get_fundamentals(
            query(table).filter(table.symbol.in_(symbols)).limit(ROW_LIMIT),
            statDate=period, latest=False)
        if not isinstance(raw, pd.DataFrame):
            raise TypeError("财务接口未返回DataFrame")
        if len(raw) >= ROW_LIMIT:
            if len(symbols) == 1:
                raise RuntimeError("单股票仍触及行数上限：" + symbols[0])
            mid = len(symbols) // 2
            return pd.concat([fetch(table, table_name, period, symbols[:mid]),
                              fetch(table, table_name, period, symbols[mid:])], ignore_index=True)
        # 保留全部原始列；重复列加后缀，不静默删除。
        counts = {}
        columns = []
        for column in raw.columns:
            counts[column] = counts.get(column, 0) + 1
            columns.append(column if counts[column] == 1 else
                           column + "__duplicate" + str(counts[column]))
        raw = raw.copy()
        raw.columns = columns
        symbol_col = field(raw, table_name, "symbol")
        period_col = field(raw, table_name, "stat_date")
        if symbol_col not in raw.columns or period_col not in raw.columns:
            raise RuntimeError("报告期查询字段结构变化：" + str(columns[:6]))
        if not set(raw[symbol_col].astype(str)).issubset(set(symbols)):
            raise RuntimeError("返回了请求范围外的股票")
        if "q" in period:
            year, quarter = period.split("q")
            expected = pd.Timestamp(int(year), int(quarter) * 3, 1) + pd.offsets.MonthEnd(0)
        else:
            expected = pd.Timestamp(int(period), 12, 31)
        if not pd.to_datetime(raw[period_col]).eq(expected).all():
            raise RuntimeError("返回报告期与请求不一致：" + period)
        return raw

    # 新单季度接口先取一只股票三个季度，结构不符即停，避免全量后才发现问题。
    for period in PERIODS:
        sample = fetch(income_sq, "income_sq", period, ["600519.SH"])
        if sample.empty:
            raise RuntimeError("单季度利润表预检为空：" + period)
        field(sample, "income_sq", "id")
        field(sample, "income_sq", "report_date")
        print("income_sq预检：", period, sample.shape, sample.iloc[:, :5].to_string(index=False))
    print("计划代码数：", len(SYMBOLS), "其中无行情代码：", len(NO_PRICE_SYMBOLS))
    for table_name, table in [("balance", balance), ("income_sq", income_sq)]:
        for period in PERIODS:
            frames = []
            for offset in range(0, len(SYMBOLS), BATCH_SIZE):
                batch = SYMBOLS[offset:offset + BATCH_SIZE]
                raw = fetch(table, table_name, period, batch)
                symbol_col = field(raw, table_name, "symbol")
                missing = sorted(set(batch) - set(raw[symbol_col].astype(str)))
                # 缺失逐只复查，避免将批量截断误认为无数据；仍空则明确记录。
                extra = []
                for symbol in missing:
                    single = fetch(table, table_name, period, [symbol])
                    if single.empty:
                        audit.append({"table": table_name, "period": period,
                                      "symbol": symbol, "status": "empty_after_single_retry"})
                    else:
                        extra.append(single)
                        audit.append({"table": table_name, "period": period,
                                      "symbol": symbol, "status": "recovered_single_retry"})
                if extra:
                    raw = pd.concat([raw] + extra, ignore_index=True)
                frames.append(raw)
                print(table_name, period, "批次", offset // BATCH_SIZE + 1,
                      "/", (len(SYMBOLS) + BATCH_SIZE - 1) // BATCH_SIZE,
                      "行数", len(raw), "累计请求", calls[0])
            frame = pd.concat(frames, ignore_index=True)
            id_col = field(frame, table_name, "id")
            report_col = field(frame, table_name, "report_date")
            duplicates = int(frame.duplicated([symbol_col, id_col]).sum())
            multi = int(frame[symbol_col].value_counts().gt(1).sum())
            dates = pd.to_datetime(frame[report_col])
            frame.attrs = {
                "source": "SuperMind", "query_mode": "statDate", "latest": False,
                "period": period, "table": table_name,
                "value_basis": "platform single-quarter income" if table_name == "income_sq" else "quarter-end balance sheet",
                "planned_symbols": SYMBOLS, "no_price_symbols": NO_PRICE_SYMBOLS,
                "price_window": ["2026-05-06", "2026-09-11"],
                "duplicate_symbol_id_rows": duplicates, "multiple_row_symbols": multi,
                "report_date_missing": int(dates.isna().sum()),
                "report_date_after_window": int(dates.gt(pd.Timestamp("2026-09-11")).sum()),
                "missing_symbols": sorted(set(SYMBOLS) - set(frame[symbol_col].astype(str))),
                "limitations": "Not all historical revisions; report_date not independently verified; no daily factor alignment; preserve missing values and duplicate records.",
            }
            save(frame, table_name + "_" + period)
            save(pd.DataFrame(audit, columns=["table", "period", "symbol", "status"]), table_name + "_" + period + "_coverage_audit")
            print("本期无返回代码：", len(frame.attrs["missing_symbols"]),
                  "重复股票-ID行：", duplicates, "多记录股票：", multi)
            del frames, frame
    print("\n拉取及文件回读结束。缺失和历史版本限制尚需评估，不能视为全市场无缺口。")
    total = sum(file_sizes.values())
    print("成品总大小MiB：", round(total / 1024 ** 2, 3), "；每个文件均不超过45MiB。")
    if total > 90 * 1024 ** 2:
        print("总量超过90MiB，请分日下载；不能同日下载全部文件。")
    print("单文件大小已核验；当前账号剩余额度未知，小文件也可能被拒绝。")
    print("请下载下列文件：")
    for name in dict.fromkeys(outputs):
        print(name)
    return list(dict.fromkeys(outputs))


financial_files = pull_financials()


### 补充拉取

In [ ]:
# SuperMind Python 3.8；已登录，研究API由平台预置。
import pandas as pd

START_DATE = "20260501"
END_DATE = "20260911"
BATCH_SIZE = 100
MAX_FILE_BYTES = 45 * 1024 ** 2

tag = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
prefix = "ths_market_cap_daily_{}_{}_{}".format(
    START_DATE, END_DATE, tag
)
temp = prefix + ".temp.pkl.gz"
outputs = []


def compress_frame(frame):
    """压缩并回读，随后彻底清理本次临时文件。"""
    try:
        frame.to_pickle(
            temp,
            compression={"method": "gzip", "compresslevel": 9},
            protocol=4,
        )
        restored = pd.read_pickle(temp, compression="gzip")
        assert restored.equals(frame)
        assert restored.attrs == frame.attrs
        return read_file(temp)
    finally:
        if temp in list_file("", isfile=True):
            remove_file(temp, trash=False)
        assert temp not in list_file("", isfile=True)


def save_parts(frame):
    """每个分片都是完整的gzip-pickle，可直接读取。"""
    payload = compress_frame(frame)

    if len(payload) > MAX_FILE_BYTES:
        middle = len(frame) // 2
        if middle == 0:
            raise RuntimeError("单行压缩后仍超过文件大小上限")
        del payload
        save_parts(frame.iloc[:middle].reset_index(drop=True))
        save_parts(frame.iloc[middle:].reset_index(drop=True))
        return

    name = "{}_part{:03d}.pkl.gz".format(prefix, len(outputs) + 1)
    assert name not in list_file("", isfile=True)

    try:
        assert write_file(name, payload) == len(payload)
        assert read_file(name) == payload
    except BaseException:
        if name in list_file("", isfile=True):
            remove_file(name, trash=False)
        raise

    outputs.append({
        "file": name,
        "rows": len(frame),
        "bytes": len(payload),
    })
    print("已保存：{}，{:.3f} MiB".format(
        name, len(payload) / 1024 ** 2
    ))


# 小文件预检：先验证压缩、平台写入、回读与清理。
probe_name = prefix + "_probe.pkl.gz"
try:
    probe_payload = compress_frame(pd.DataFrame({"test": [1]}))
    assert write_file(probe_name, probe_payload) == len(probe_payload)
    assert read_file(probe_name) == probe_payload
finally:
    if probe_name in list_file("", isfile=True):
        remove_file(probe_name, trash=False)
assert probe_name not in list_file("", isfile=True)

# 每天使用当日股票目录，不用最新股票名单替代历史名单。
trade_days = pd.DatetimeIndex(
    get_trade_days(START_DATE, END_DATE)
).normalize()

daily_frames = []

for day_number, day in enumerate(trade_days, 1):
    date = day.strftime("%Y%m%d")
    securities = get_all_securities("stock", date)
    symbols = sorted({
        str(symbol)
        for symbol in securities.index
        if str(symbol).endswith((".SH", ".SZ", ".BJ"))
    })

    batches = []
    for start in range(0, len(symbols), BATCH_SIZE):
        batch = symbols[start:start + BATCH_SIZE]

        raw = get_fundamentals(
            query(valuation.symbol, valuation.market_cap)
            .filter(valuation.symbol.in_(batch))
            .limit(1000),
            date=date,
        )
        batches.append(
            raw[["valuation_symbol", "valuation_market_cap"]]
        )

    returned = pd.concat(batches, ignore_index=True).rename(columns={
        "valuation_symbol": "symbol",
        "valuation_market_cap": "market_cap",
    })

    # 未返回市值的代码仍保留为空，不用其他日期补值。
    daily = pd.DataFrame({"symbol": symbols}).merge(
        returned,
        on="symbol",
        how="left",
        validate="one_to_one",
    )
    daily["time"] = day
    daily["code"] = (
        daily["symbol"]
        .str.replace(".SH", ".XSHG", regex=False)
        .str.replace(".SZ", ".XSHE", regex=False)
        .str.replace(".BJ", ".XBSE", regex=False)
    )
    daily = daily[["time", "code", "market_cap"]]
    daily_frames.append(daily)

    print(
        "{}/{} {}：{}只，市值非空{}只，空值{}只".format(
            day_number, len(trade_days), date, len(daily),
            daily["market_cap"].notna().sum(),
            daily["market_cap"].isna().sum(),
        )
    )

market_cap_daily = (
    pd.concat(daily_frames, ignore_index=True)
    .sort_values(["time", "code"])
    .reset_index(drop=True)
)
del daily_frames

market_cap_daily["code"] = market_cap_daily["code"].astype("category")
market_cap_daily.attrs = {
    "source": "SuperMind valuation.market_cap",
    "market_cap_unit": "CNY",
    "start_date": START_DATE,
    "end_date": END_DATE,
    "frequency": "daily",
    "universe": "daily Shanghai Shenzhen Beijing stock lists",
}

assert not market_cap_daily.duplicated(["time", "code"]).any()
assert market_cap_daily["time"].nunique() == len(trade_days)

save_parts(market_cap_daily)

delivery = pd.DataFrame(outputs)
assert delivery["rows"].sum() == len(market_cap_daily)

# 下载批次按90 MiB计划；实际剩余额度以账号为准。
download_batch, used_bytes = 1, 0
download_batches = []
for size in delivery["bytes"]:
    if used_bytes + size > 90 * 1024 ** 2:
        download_batch += 1
        used_bytes = 0
    download_batches.append(download_batch)
    used_bytes += size

delivery["MiB"] = delivery["bytes"] / 1024 ** 2
delivery["download_batch"] = download_batches

print("\n完成：{}行，{}个交易日，{}个文件，共{:.3f} MiB".format(
    len(market_cap_daily),
    market_cap_daily["time"].nunique(),
    len(delivery),
    delivery["MiB"].sum(),
))
print(delivery.to_string(index=False))
print("\n每片均可直接pd.read_pickle；多片按part序号读取后pd.concat。")
print("全部成品已回读核对；实际下载额度以账号为准。")

In [ ]:
import pandas as pd

SNAPSHOT_DATE = "20260911"
OUTPUT_FILE = "ths_stock_code_name_20260911.pkl.gz"

stock_name_map = (
    get_all_securities("stock", SNAPSHOT_DATE)
    .reset_index()[["symbol", "display_name"]]
    .rename(columns={
        "symbol": "ths_code",
        "display_name": "name",
    })
)

# 转换为当前本地数据使用的代码格式
stock_name_map["code"] = (
    stock_name_map["ths_code"].astype(str)
    .str.replace(r"\.SH$", ".XSHG", regex=True)
    .str.replace(r"\.SZ$", ".XSHE", regex=True)
    .str.replace(r"\.BJ$", ".XBSE", regex=True)
)

stock_name_map = (
    stock_name_map[["code", "name", "ths_code"]]
    .sort_values("code")
    .reset_index(drop=True)
)

assert stock_name_map["code"].is_unique
assert stock_name_map["name"].notna().all()

# 使用当前环境已经验证可用的 pandas gzip-pickle 写法
stock_name_map.to_pickle(
    OUTPUT_FILE,
    compression="gzip",
    protocol=4,
)

# 回读核验
stock_name_map_check = pd.read_pickle(
    OUTPUT_FILE,
    compression="gzip",
)
assert stock_name_map_check.equals(stock_name_map)

print("股票数量：", len(stock_name_map))
print("输出文件：", OUTPUT_FILE)
print("文件大小：{:.2f} MiB".format(len(read_file(OUTPUT_FILE)) / 1024 ** 2))
display(stock_name_map.head(10))

In [ ]:
# SuperMind Python 3.8；独立单元格，一次执行。拉取同花顺L2行业日行情。
# industry_symbol（如T1203）用于查询成分；industry_thscode（如881273.TI）用于拉行情。
import pandas as pd

START_DATE = "20260501"
END_DATE = "20260911"
INDUSTRY_TYPE = "industryid2"
BATCH_SIZE = 30
MAX_FILE_BYTES = 45 * 1024 ** 2
PRICE_FIELDS = ["open", "high", "low", "close", "volume", "turnover"]

tag = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
prefix = "ths_industry_l2_daily_{}_{}_{}".format(START_DATE, END_DATE, tag)
temp_file = prefix + ".temp.pkl.gz"
output_file = prefix + ".pkl.gz"


def remove_owned_file(path):
    if path in list_file("", isfile=True):
        remove_file(path, trash=False)
    if path in list_file("", isfile=True):
        raise RuntimeError("文件未彻底删除：" + path)


def fetch_prices(symbols):
    raw = get_price(
        symbols, START_DATE, END_DATE, "1d", PRICE_FIELDS,
        skip_paused=False, fq=None, bar_count=0, is_panel=False,
    )
    if isinstance(raw, pd.DataFrame) and len(symbols) == 1:
        raw = {symbols[0]: raw}
    if not isinstance(raw, dict):
        raise TypeError("多行业行情返回类型异常：" + type(raw).__name__)
    return raw


required_apis = (get_trade_days, get_industry_relate, get_price,
                 read_file, write_file, list_file, remove_file)
trade_days = pd.DatetimeIndex(get_trade_days(START_DATE, END_DATE)).normalize().sort_values()
if trade_days.empty or trade_days.has_duplicates:
    raise RuntimeError("交易日历为空或重复")

# 研究区间内逐日读取目录，取所有曾经有效的L2行业行情代码。
catalog_frames = []
for day in trade_days:
    date = day.strftime("%Y%m%d")
    catalog = get_industry_relate(date=date, types=INDUSTRY_TYPE, fields=None).reset_index()
    required_columns = {"industry_name", "industry_symbol", "industry_thscode"}
    missing_columns = required_columns.difference(catalog.columns)
    if missing_columns:
        raise RuntimeError("行业目录缺少字段：{}；日期：{}".format(
            sorted(missing_columns), date))
    catalog = catalog[["industry_name", "industry_symbol", "industry_thscode"]].copy()
    catalog["catalog_date"] = day
    catalog_frames.append(catalog)

catalog_history = pd.concat(catalog_frames, ignore_index=True).dropna(
    subset=["industry_name", "industry_symbol", "industry_thscode"]
)
del catalog_frames
catalog_history = catalog_history.sort_values("catalog_date")
catalog_latest = catalog_history.drop_duplicates("industry_thscode", keep="last")
catalog_latest = catalog_latest.sort_values("industry_thscode").reset_index(drop=True)
if catalog_latest["industry_thscode"].duplicated().any():
    raise RuntimeError("行业行情代码重复")
industry_codes = catalog_latest["industry_thscode"].astype(str).tolist()
if not industry_codes:
    raise RuntimeError("研究区间内没有有效L2行业行情代码")

name_map = catalog_latest.set_index("industry_thscode")["industry_name"].astype(str).to_dict()
symbol_map = catalog_latest.set_index("industry_thscode")["industry_symbol"].astype(str).to_dict()
frames = []
empty_after_retry = []

for start in range(0, len(industry_codes), BATCH_SIZE):
    batch = industry_codes[start:start + BATCH_SIZE]
    raw = fetch_prices(batch)
    for code in batch:
        item = raw.get(code)
        if item is None or item.empty:
            retry = fetch_prices([code])
            item = retry.get(code)
        if item is None or item.empty:
            empty_after_retry.append(code)
            continue
        missing_fields = sorted(set(PRICE_FIELDS).difference(item.columns))
        if missing_fields:
            raise RuntimeError("{} 行情缺少字段：{}".format(code, missing_fields))
        part = item[PRICE_FIELDS].copy()
        part.index.name = "time"
        part = part.reset_index().rename(columns={"turnover": "money"})
        part["time"] = pd.to_datetime(part["time"]).dt.normalize()
        if part["time"].duplicated().any():
            raise RuntimeError("同一行业存在重复日线：" + code)
        if not part["time"].isin(trade_days).all():
            raise RuntimeError("行业行情日期超出请求交易日范围：" + code)
        if (part[["volume", "money"]] < 0).any().any():
            raise RuntimeError("行业成交量或成交额为负：" + code)
        part.insert(1, "industry_name", name_map[code])
        part.insert(2, "industry_symbol", symbol_map[code])
        part.insert(3, "industry_thscode", code)
        part["return_1d"] = part["close"].pct_change(fill_method=None)
        frames.append(part)
    print("行情批次 {}/{}；累计空返回 {} 个行业".format(
        start // BATCH_SIZE + 1,
        (len(industry_codes) + BATCH_SIZE - 1) // BATCH_SIZE,
        len(empty_after_retry),
    ))

if not frames:
    raise RuntimeError("所有L2行业行情均为空")
industry_daily = pd.concat(frames, ignore_index=True)
del frames
industry_daily = industry_daily.sort_values(
    ["industry_thscode", "time"]
).reset_index(drop=True)
if industry_daily.duplicated(["time", "industry_thscode"]).any():
    raise RuntimeError("行业—交易日主键重复")

coverage_rows = []
for code in industry_codes:
    returned_days = pd.DatetimeIndex(
        industry_daily.loc[industry_daily["industry_thscode"].eq(code), "time"].unique()
    ).sort_values()
    missing_days = trade_days.difference(returned_days)
    coverage_rows.append({
        "industry_name": name_map[code],
        "industry_symbol": symbol_map[code],
        "industry_thscode": code,
        "expected_trade_days": len(trade_days),
        "returned_trade_days": len(returned_days),
        "missing_trade_days": [d.strftime("%Y%m%d") for d in missing_days],
        "status": "empty_after_single_retry" if code in empty_after_retry else "returned",
    })
coverage_audit = pd.DataFrame(coverage_rows)

for column in ["industry_name", "industry_symbol", "industry_thscode"]:
    industry_daily[column] = industry_daily[column].astype("category")
industry_daily.attrs = {
    "source": "SuperMind get_price",
    "industry_type": INDUSTRY_TYPE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "frequency": "1d",
    "fq": None,
    "money_unit": "CNY",
    "planned_industry_codes": industry_codes,
    "empty_after_single_retry": empty_after_retry,
    "coverage_audit": coverage_rows,
    "return_1d": "close/previous returned close-1; first returned day is null",
    "created_at": pd.Timestamp.now().isoformat(),
}

if temp_file in list_file("", isfile=True) or output_file in list_file("", isfile=True):
    raise RuntimeError("本次唯一文件名已存在，不覆盖")
try:
    industry_daily.to_pickle(
        temp_file, compression={"method": "gzip", "compresslevel": 9}, protocol=4
    )
    restored = pd.read_pickle(temp_file, compression="gzip")
    if not restored.equals(industry_daily) or restored.attrs != industry_daily.attrs:
        raise RuntimeError("压缩回读与原数据不一致")
    payload = read_file(temp_file)
finally:
    remove_owned_file(temp_file)

if len(payload) > MAX_FILE_BYTES:
    raise RuntimeError("行业行情成品超过45MiB，需要改为按行业顺序分片")
try:
    written = write_file(output_file, payload)
    checked = read_file(output_file)
    if written != len(payload) or checked != payload:
        raise RuntimeError("成品写入核对失败")
except BaseException:
    remove_owned_file(output_file)
    raise

print("完成：{}行，{}个L2行业，{}个交易日，{:.3f} MiB".format(
    len(industry_daily), industry_daily["industry_thscode"].nunique(),
    industry_daily["time"].nunique(), len(payload) / 1024 ** 2,
))
print("输出文件：", output_file)
print("重试后仍为空的行业行情代码：", empty_after_retry)
display(coverage_audit)
display(industry_daily.head())